# Bubble Tracker GUI

The Bubble Tracker GUI is an interactive Python/Jupyter-based application developed to perform automated bubble detection, tracking, and measurement from CNTR video data and TIFF image sequences. The tool combines a Detectron2 Mask R-CNN instance-segmentation model, a secondary CNN bubble-verification model, and a Kalman filter with assignment-based multi-object tracking.

The software supports both still and spinning experimental configurations. Still mode uses a constant user-specified frame rate, while spinning mode extracts timing from on-screen timestamps using template matching and can account for alternating frame orientation during rotation.

The application produces annotated videos, predicted mask videos, frame-resolved tracking CSV files, per-bubble/overall average CSV files, and optional debug outputs. When a spatial calibration is provided, the code converts pixel measurements to physical units and computes dimensionless parameters including Reynolds number, Eötvös number, Weber number, Froude number, Galilei number, Morton number, and Bond number.

## Main Capabilities

- Processes either video files or TIFF image folders
- Loads Mask R-CNN weights with the matching `config.yaml`
- Uses CNN verification to reduce false bubble detections
- Tracks bubbles using Kalman filtering and assignment logic
- Supports still-mode and spinning-mode analysis
- Performs timestamp-based timing in spinning mode
- Applies top-center bar alignment for spinning-mode stabilization
- Exports annotated videos and binary predicted-mask videos
- Exports `tracked.csv` and `averages.csv`
- Computes bubble diameter, centroid position, velocity, and dimensionless parameters
- Provides optional debug videos, logs, and assignment diagnostics

## Required Inputs

- Input video file or TIFF image folder
- Mask R-CNN model weights (`.pth` or `.pt`)
- Matching `config.yaml` located in the same folder as the Mask R-CNN weights
- CNN classifier weights (`.pth` or `.pt`)
- Output folder
- Analysis settings such as FPS, scale, diameter limits, fluid type, gas type, and operating mode

## Main Outputs

- `{prefix}_filtered.mp4`  
  Final annotated video showing accepted bubble tracks.

- `{prefix}_pred_mask.mp4`  
  Binary mask video showing predicted bubble regions.

- `{prefix}_tracked.csv`  
  Frame-resolved bubble measurements including centroid position, diameter, velocity, and dimensionless parameters when scale is provided.

- `{prefix}_averages.csv`  
  Per-bubble and overall averaged quantities, along with the GUI settings used for the run.

- Optional debug outputs  
  Additional videos, debug logs, and tracking-association data used for troubleshooting and validation.

## Operating Modes

### Still Mode

Still mode assumes uniform frame spacing based on the user-specified FPS. This mode is used for standard non-rotating BUBBLINA-style experiments.

### Spinning Mode

Spinning mode is intended for rotating-apparatus data. It extracts frame timing from timestamp digits using template matching, optionally resets tracks by burst, flips alternating burst frames, and uses a top-center bar alignment routine to stabilize the apparatus position between frames.

## Notes

This notebook requires Detectron2, PyTorch, OpenCV, NumPy, Pandas, SciPy, FilterPy, Pillow, and Tkinter. Model files must be available locally before running the GUI.

The spinning-mode functionality is an experimental extension beyond the primary thesis methodology and should be documented separately in `Documentation/spinning_mode.md`.

In [ ]:
# ============================================================
# Bubble Evaluation Tool (Appendix Code)
# Purpose: Automated bubble detection, tracking, and measurement from CNTR video data
# Inputs: CNTR video + Mask R-CNN weights + CNN weights + analysis settings (scale, fluids, mode)
# Outputs: filtered/annotated video(s), tracked.csv, averages.csv (+ optional debug logs)
# Methods: Mask R-CNN segmentation, CNN verification, Kalman + assignment tracking,
#          post-processing and dimensionless parameter calculations
# Note: Running requires Detectron2, PyTorch, FilterPy, and trained model weights.
# ============================================================

import os
import cv2
import numpy as np
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pandas as pd
import math, csv, warnings
from math import pi

import glob
import re

# Detectron2 imports
from detectron2.config          import get_cfg
from detectron2.engine          import DefaultPredictor
from detectron2.utils.logger    import setup_logger
from detectron2                 import model_zoo

# Tracking imports
from filterpy.kalman            import KalmanFilter
from scipy.optimize             import linear_sum_assignment

# CNN imports
import torch
from torchvision import transforms
from PIL import Image

DEBUG_LOG = None   # global handle that will open per run

def dprint(*args, **kwargs):
    global DEBUG_LOG
    if DEBUG_LOG is not None:
        print(*args, **kwargs, file=DEBUG_LOG)

# ---------------- Fluids / physics ----------------
# Note: Dimensionless numbers are computed using consistent English units with gc (G_C).
IQ_PROPS = {
    "Water": {"mu": 0.0000001412232, "sigma": 0.000416, "rho": 0.03605},   # mu: lbf*s/in^2, sigma: lbf/in, rho: lbm/in^3
    "SPT3" : {"mu": 0.000005076321, "sigma": 0.000406, "rho": 0.10838},
}
LIQ_PROPS = IQ_PROPS
GAS_RHO_FT3 = {"Air": 0.07487, "Nitrogen": 0.07210}  # lbm/ft^3

G_C       = 386.4   # (lbm*in)/(lbf*s^2)
G0_IN_S2  = 386.4   # in/s^2
RADIUS_IN = 3.125   # in

# ---------------- Tunables ----------------
IOU_WEIGHT_DEFAULT   = 3.0   # association balance

# Merge/split bookkeeping
IOU_MERGE_THRESH     = 0.20
IOU_SPLIT_THRESH     = 0.20
IOU_REASSOC_THRESH   = 0.30

# Confirmation & coasting
CONFIRM_HITS         = 3
MAX_COAST            = 14
MAX_OCCLUDE_AGE      = 30

# Viz clustering
MERGE_LABEL_MIN_STREAK = 2
CLUSTER_IOU_THR        = 0.75
CLUSTER_CENTER_FRAC    = 0.55

# Timestamp template match (spin mode)
TEMPLATE_DIR = "templates"
X0, Y0, X1, Y1 = 32, -30, 100, None
MATCH_THRESH = 0.85

digit_templates = {}
for d in "0123456789":
    fn = os.path.join(TEMPLATE_DIR, f"{d}.png")
    tpl = cv2.imread(fn, cv2.IMREAD_GRAYSCALE)
    if tpl is None:
        raise FileNotFoundError(f"Could not load template {fn}")
    _, bw = cv2.threshold(tpl, 127, 255, cv2.THRESH_BINARY_INV)
    digit_templates[d] = bw

def read_timestamp_tpl(frame):
    H, W = frame.shape[:2]
    y1 = H if Y1 is None else Y1
    crop = frame[H+Y0:H+y1, X0:X1]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    res_maps = {d: cv2.matchTemplate(bw, tpl, cv2.TM_CCOEFF_NORMED)
                for d, tpl in digit_templates.items()}
    candidates = []
    w_min = min(tpl.shape[1] for tpl in digit_templates.values())
    Wb = bw.shape[1]
    for x in range(Wb - w_min + 1):
        best_d, best_s = None, -1.0
        for d, res in res_maps.items():
            if x < res.shape[1]:
                s = res[:, x].max()
                if s > best_s:
                    best_s, best_d = s, d
        if best_s >= MATCH_THRESH:
            candidates.append((x, best_d))
    if not candidates:
        return None
    candidates.sort(key=lambda t: t[0])
    out, last_x = [], -999
    for x, d in candidates:
        w = digit_templates[d].shape[1]
        if x - last_x > w // 2:
            out.append((x, d))
            last_x = x
    out.sort(key=lambda t: t[0])
    digits = "".join(d for _, d in out)
    if len(digits) < 4:
        return None
    return digits[:-3] + "." + digits[-3:]

# Colors
COLOR_MAP = {
    "Red":    (0, 0, 255),
    "Orange": (0,165,255),
    "Yellow": (0,255,255),
    "Green":  (0,255,0),
    "Blue":   (255,0,0),
    "Indigo": (130,0,75),
    "Purple": (240,80,160),
    "Pink":   (147,20,255),
}

# -------- detection / gating / smoothing ----------
DET_SCORE_THRESH     = 0.05     # 0.50 
CLS_ENTER            = 0.40     # prob to allow spawn
CLS_STAY             = 0.30     # prob to remain visible/valid

EMA_D_ALPHA          = 0.30
MAX_DIA_GROWTH       = 3.0
MAX_DIA_SHRINK       = 0.40
UPWARD_WEIGHT        = 0.0

GATE_MAHAL2          = 18.0
MIN_IOU_FOR_ASSIGN   = 0.05

ROI_SIZE = 128
DEFAULT_BURST_FRAMES = 8
DEFAULT_GAP_THRESH_MS = 30.0
DEFAULT_ABSOLUTE_CAP_MS = 200.0
DEFAULT_IN_BURST_THRESH_MS = 0.30
OUTPUT_FPS = 30.0

COUNT_PLANE_Y_FRAC = 0.50   # plane at 50% of frame height (0.0 top, 1.0 bottom)
COUNT_DIRECTION    = "up"   # "up" (rising, cy decreases), "down", or "both"


to_tensor_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ----------------------- helpers -----------------------

def mask_iou(a_bool, b_bool):
    inter = np.logical_and(a_bool, b_bool).sum()
    union = np.logical_or(a_bool, b_bool).sum()
    if union == 0:
        return 1.0 if inter == 0 else 0.0
    return float(inter) / float(union)

def mahalanobis2(trk, z):
    H = trk.kf.H
    S = H @ trk.kf.P @ H.T + trk.kf.R
    y = np.array(z, dtype=float).reshape(2,1) - H @ trk.kf.x
    return float((y.T @ np.linalg.inv(S) @ y).ravel())

def choose_mask(prev_mask, new_mask,
                iou_keep_prev=0.25,     
                area_ratio_min=0.55,
                area_ratio_max=1.90):
    prev_area = float(prev_mask.sum())
    new_area  = float(new_mask.sum())
    if prev_area > 0 and new_area > 0:
        iou = mask_iou(prev_mask, new_mask)
        ratio = new_area / prev_area
        if (iou < iou_keep_prev) and (ratio < area_ratio_min or ratio > area_ratio_max):
            return prev_mask
    dprint(f"choose_mask: iou={iou:.3f} ratio={ratio:.2f} -> {'KEEP_PREV' if ((iou < iou_keep_prev) and (ratio < area_ratio_min or ratio > area_ratio_max)) else 'USE_NEW'}")        
    return new_mask

def area_to_diam(mask_bool):
    a = float(mask_bool.sum())
    return 2.0*np.sqrt(a/pi) if a > 0 else 0.0

def diameter_jump_ok(prev_d, new_d):
    if prev_d <= 0 or new_d <= 0:
        return True
    ratio = new_d / prev_d
    return (ratio <= MAX_DIA_GROWTH) and (ratio >= MAX_DIA_SHRINK)

def mask_pairwise_nms(dm, score, iou_dup=0.86):
    n = len(dm)
    keep = np.ones(n, dtype=bool)
    order = np.argsort(-np.asarray(score, float))
    for a in range(n):
        i = order[a]
        if not keep[i]:
            continue
        for b in range(a+1, n):
            j = order[b]
            if not keep[j]:
                continue
            if mask_iou(dm[i], dm[j]) >= iou_dup:
                if score[i] >= score[j]:
                    keep[j] = False
                else:
                    keep[i] = False
                    break
    return keep

def assign_to_members(member_idxs, det_idxs, dc, dm, tracks, iou_weight=IOU_WEIGHT_DEFAULT, char_dist=12.0):
    if not member_idxs or not det_idxs: return []
    M, N = len(member_idxs), len(det_idxs)
    cost = np.full((M, N), 1e6, float)
    for a, ti in enumerate(member_idxs):
        px, py, _, _ = tracks[ti].state()
        for b, dj in enumerate(det_idxs):
            cx, cy = dc[dj]
            d = np.hypot(px - cx, py - cy)
            iou = mask_iou(tracks[ti].mask, dm[dj])
            up_pen = UPWARD_WEIGHT * max(0.0, cy - py)
            cost[a, b] = (d / max(char_dist, 1.0)) + iou_weight*(1.0 - iou) + up_pen
    r, c = linear_sum_assignment(cost)
    return [(member_idxs[i], det_idxs[j]) for i, j in zip(r, c) if cost[i, j] < 1e6]

def crop_center_square(frame, cx, cy, size=ROI_SIZE):
    H, W = frame.shape[:2]
    half = size // 2
    x0, y0 = int(round(cx)) - half, int(round(cy)) - half
    x1, y1 = x0 + size, y0 + size
    patch = np.zeros((size, size, 3), dtype=frame.dtype)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(W, x1), min(H, y1)
    dx0, dy0 = sx0 - x0, sy0 - y0
    dx1, dy1 = dx0 + (sx1 - sx0), dy0 + (sy1 - sy0)
    if sy1 > sy0 and sx1 > sx0 and dy1 > 0:
        patch[dy0:dy1, dx0:dx1] = frame[sy0:sy1, sx0:sx1]
    return Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))

def find_center_bar_midpoint(frame):
    """
    Find the geometric x-center of the upper center rod.

    This version finds the full dark vertical band, then returns the midpoint
    between its left and right edges. It is better than picking the darkest
    column, because lighting can make one side of the rod darker than the other.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    # Adjust only these four values if needed.
    y0 = int(0.02 * H)
    y1 = int(0.10 * H)
    x0 = int(0.30 * W)
    x1 = int(0.70 * W)

    roi = gray[y0:y1, x0:x1]
    if roi.size == 0:
        return None

    blur = cv2.GaussianBlur(roi, (5, 5), 0)

    # Column intensity profile.
    # Dark rod columns have lower average intensity.
    col_mean = np.mean(blur, axis=0)

    # Smooth across x so texture/noise does not dominate.
    col_mean_smooth = cv2.GaussianBlur(
        col_mean.reshape(1, -1).astype(np.float32),
        (1, 21),
        0
    ).ravel()

    roi_w = x1 - x0
    roi_center = roi_w / 2.0

    # Use a threshold relative to this ROI.
    # Dark columns are below this threshold.
    min_val = float(np.min(col_mean_smooth))
    max_val = float(np.max(col_mean_smooth))

    if max_val - min_val < 5:
        dprint("Rod profile too flat; no reliable rod found")
        return None

    # This selects the dark rod band, not just the single darkest column.
    thresh = min_val + 0.45 * (max_val - min_val)
    dark_cols = col_mean_smooth < thresh

    # Find connected dark-column runs.
    runs = []
    start = None

    for i, is_dark in enumerate(dark_cols):
        if is_dark and start is None:
            start = i
        elif not is_dark and start is not None:
            runs.append((start, i - 1))
            start = None

    if start is not None:
        runs.append((start, len(dark_cols) - 1))

    candidates = []

    for a, b in runs:
        width = b - a + 1
        center = 0.5 * (a + b)

        # Rod should be a substantial vertical band, not a thin noise stripe.
        if width < 20:
            continue

        # It should be near the center of the ROI.
        center_penalty = abs(center - roi_center)

        # Prefer wide centered dark bands.
        score = width - 0.5 * center_penalty
        candidates.append((score, a, b, width, center))

    dprint(f"Rod dark-column runs: {runs}")
    dprint(f"Rod candidates: {candidates}")

    if not candidates:
        return None

    candidates.sort(key=lambda t: t[0], reverse=True)
    _, a, b, width, center = candidates[0]

    rod_center_x = x0 + center

    dprint(
        f"Rod band selected: left={x0+a:.2f}, right={x0+b:.2f}, "
        f"width={width}, center={rod_center_x:.2f}"
    )

    return float(rod_center_x)

def cluster_representatives(visible, iou_thr=CLUSTER_IOU_THR, center_frac=CLUSTER_CENTER_FRAC):
    n = len(visible)
    if n <= 1:
        return visible[:]
    adj = [set() for _ in range(n)]
    centers = [np.array(tr.state()[:2], float) for tr in visible]
    diams   = [float(getattr(tr, 'diam_sm', 0.0)) for tr in visible]
    for i in range(n):
        for j in range(i+1, n):
            iou = mask_iou(visible[i].mask, visible[j].mask)
            d   = np.linalg.norm(centers[i] - centers[j])
            close = d <= center_frac * max(1.0, min(diams[i], diams[j]))
            if (iou >= iou_thr) or close:
                adj[i].add(j); adj[j].add(i)
    seen = [False]*n; clusters = []
    for i in range(n):
        if seen[i]: continue
        comp = [i]; seen[i] = True
        stack = [i]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
                    comp.append(v)
        clusters.append(comp)
    reps = []
    for comp in clusters:
        best_tr, best_s = None, -1e9
        for idx in comp:
            tr = visible[idx]
            s = 2.0*tr.hit_streak + 1.0*tr.cnn_p + 0.3*(tr.diam_sm) - 0.8*tr.time_since_update - (1.0 if tr.occluded else 0.0)
            if s > best_s:
                best_s = s; best_tr = tr
        reps.append(best_tr)
    return reps
    
def natural_key(path):
    # natural sort for filenames: Img2 before Img10
    base = os.path.basename(path)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', base)]

def list_tif_files(folder):
    patterns = ["*.tif", "*.tiff", "*.TIF", "*.TIFF"]

    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(folder, pat)))

    files = list(dict.fromkeys(os.path.normpath(f) for f in files))

    files = sorted(files, key=natural_key)
    return files

# ----------------------- GUI app -----------------------
class BubbleTrackerApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Bubble Tracker GUI")
        self._build_ui()

    def _build_ui(self):
        pad, row = {"padx":4, "pady":2}, 0
        def add(lbl, ent):
            nonlocal row
            ttk.Label(self, text=lbl).grid(row=row, column=0, sticky="e", **pad)
            ent.grid(row=row, column=1, **pad)
            row += 1

        # Inputs
        self.vid = ttk.Entry(self, width=42);        add("Input Video:", self.vid)
        ttk.Button(self, text="Browse…", command=lambda: self._browse(self.vid, [("VIDEO","*.avi *.mp4 *.mov *.mkv *.m4v")])).grid(row=row-1, column=2, **pad)

        self.img_dir = ttk.Entry(self, width=42);    add("Input TIFF Folder:", self.img_dir)
        ttk.Button(self, text="Browse…", command=lambda: self._browse_dir(self.img_dir)).grid(row=row-1, column=2, **pad)

        ttk.Label(self, text="Input Type:").grid(row=row, column=0, sticky="e", **pad)
        self.input_type = tk.StringVar(value="video")
        ttk.Radiobutton(self, text="Video", variable=self.input_type, value="video").grid(row=row, column=1, sticky="w", **pad)
        ttk.Radiobutton(self, text="TIFF folder", variable=self.input_type, value="tif").grid(row=row, column=2, sticky="w", **pad)
        row += 1

        self.mask_w = ttk.Entry(self, width=42);     add("Mask-R-CNN Weights:", self.mask_w)
        ttk.Button(self, text="Browse…", command=lambda: self._browse(self.mask_w, [("PyTorch","*.pth *.pt")])).grid(row=row-1, column=2, **pad)

        self.cnn_w = ttk.Entry(self, width=42);      add("CNN Weights:", self.cnn_w)
        ttk.Button(self, text="Browse…", command=lambda: self._browse(self.cnn_w, [("PyTorch","*.pth *.pt")])).grid(row=row-1, column=2, **pad)

        self.out_dir = ttk.Entry(self, width=42);    add("Output Folder:", self.out_dir)
        ttk.Button(self, text="Browse…", command=lambda: self._browse_dir(self.out_dir)).grid(row=row-1, column=2, **pad)

        self.prefix = ttk.Entry(self, width=14); self.prefix.insert(0, ""); add("Filename Prefix:", self.prefix)

        self.fps_e = ttk.Entry(self, width=6); self.fps_e.insert(0, "1200"); add("FPS (still mode):", self.fps_e)
        self.max_d = ttk.Entry(self, width=6); self.max_d.insert(0, "65"); add("Max assign dist (px):", self.max_d)
        self.max_m = ttk.Entry(self, width=6); self.max_m.insert(0, "5");  add("Max misses (frames):", self.max_m)
        self.ign_h = ttk.Entry(self, width=6); self.ign_h.insert(0, "0");  add("Ignore text (px):", self.ign_h)

        # bubble dia
        ttk.Label(self, text="Bubble diameter (px):").grid(row=row, column=0, sticky="e", **pad)
        dframe = ttk.Frame(self)
        self.min_diam_px = ttk.Entry(dframe, width=6); self.min_diam_px.insert(0, "5")
        self.max_diam_px = ttk.Entry(dframe, width=6); self.max_diam_px.insert(0, "240")
        ttk.Label(dframe, text="min").grid(row=0, column=0, **pad)
        self.min_diam_px.grid(row=0, column=1, **pad)
        ttk.Label(dframe, text="max").grid(row=0, column=2, **pad)
        self.max_diam_px.grid(row=0, column=3, **pad)
        dframe.grid(row=row, column=1, columnspan=2, sticky="w", **pad)
        row += 1

        # scale
        ttk.Label(self, text="Scale:").grid(row=row, column=0, sticky="e", **pad)
        scf = ttk.Frame(self)
        self.scale_in = ttk.Entry(scf, width=8); self.scale_px = ttk.Entry(scf, width=8)
        self.scale_in.insert(0, "0.5"); self.scale_px.insert(0, "100.0")
        self.scale_in.grid(row=0, column=0, **pad); ttk.Label(scf, text="Inches per").grid(row=0, column=1, **pad)
        self.scale_px.grid(row=0, column=2, **pad); ttk.Label(scf, text="pixels").grid(row=0, column=3, **pad)
        scf.grid(row=row, column=1, columnspan=2, sticky="w", **pad); row += 1

        # fluids
        ttk.Label(self, text="Liquid:").grid(row=row, column=0, sticky="e", **pad)
        self.liq_var = tk.StringVar(value="Water")
        ttk.Combobox(self, textvariable=self.liq_var,
                     values=list(IQ_PROPS.keys()), state="readonly", width=14
        ).grid(row=row, column=1, columnspan=2, sticky="w", **pad)
        row += 1

        ttk.Label(self, text="Gas:").grid(row=row, column=0, sticky="e", **pad)
        self.gas_var = tk.StringVar(value="Air")
        ttk.Combobox(self, textvariable=self.gas_var,
                     values=list(GAS_RHO_FT3.keys()), state="readonly", width=14
        ).grid(row=row, column=1, columnspan=2, sticky="w", **pad)
        row += 1

        ttk.Label(self, text="Overlay Color:").grid(row=row, column=0, sticky="e", **pad)
        self.color_var = tk.StringVar(value="Purple")
        self.color_cb = ttk.Combobox(self, textvariable=self.color_var, values=list(COLOR_MAP.keys()), state="readonly", width=14)
        self.color_cb.grid(row=row, column=1, columnspan=2, sticky="w", **pad); row += 1

        # mode
        ttk.Label(self, text="Mode:").grid(row=row, column=0, sticky="e", **pad)
        self.mode = tk.StringVar(value="still")
        ttk.Radiobutton(self, text="Still",    variable=self.mode, value="still", command=self._toggle).grid(row=row, column=1, sticky="w")
        ttk.Radiobutton(self, text="Spinning", variable=self.mode, value="spin",  command=self._toggle).grid(row=row, column=2, sticky="w"); row += 1

        self.rpm_lbl = ttk.Label(self, text="RPM (spin):")
        self.rpm_e   = ttk.Entry(self, width=8); self.rpm_e.insert(0, "0")
        self.rpm_lbl.grid(row=row, column=0, sticky="e", **pad)
        self.rpm_e.grid(row=row, column=1, sticky="w", **pad); row += 1

        self.b_lbl = ttk.Label(self, text="Frames/Burst:")
        self.b_ent = ttk.Entry(self, width=6); self.b_ent.insert(0, str(DEFAULT_BURST_FRAMES))
        self.r_chk = tk.BooleanVar(value=True); self.b_chk = ttk.Checkbutton(self, text="Reset IDs per burst", variable=self.r_chk)
        self.b_lbl.grid(row=row, column=0, **pad); self.b_ent.grid(row=row, column=1, **pad); self.b_chk.grid(row=row, column=2, **pad); row += 1

        # output options
        self.save_debug = tk.BooleanVar(value=False)   # v1, v2, v4 + debug_pairs.csv
        self.save_csvs  = tk.BooleanVar(value=True)   # tracked.csv + averages.csv
        ttk.Checkbutton(self, text="Save debug outputs",
                        variable=self.save_debug).grid(row=row, column=0, columnspan=3, sticky="w", padx=4, pady=(8,2))
        row += 1
        ttk.Checkbutton(self, text="Save measurement CSVs",
                        variable=self.save_csvs).grid(row=row, column=0, columnspan=3, sticky="w", padx=4, pady=(0,6))
        row += 1

        ttk.Button(self, text="Run Tracking", command=self._run).grid(row=row, column=0, columnspan=3, pady=(6,4)); row += 1
        self.prog = ttk.Progressbar(self, length=420, mode="determinate"); self.prog.grid(row=row, column=0, columnspan=3, pady=(4,8))
        self._toggle()

    def _browse(self, entry, types):
        fn = filedialog.askopenfilename(filetypes=types)
        if fn:
            entry.delete(0, tk.END); entry.insert(0, fn)
            # Auto-fill prefix with input video basename (no ext) if empty or different video selected
            base = os.path.splitext(os.path.basename(fn))[0]
            cur_prefix = (self.prefix.get() or "").strip()
            if not cur_prefix or cur_prefix == "output":
                self.prefix.delete(0, tk.END)
                self.prefix.insert(0, base)

    def _browse_dir(self, entry):
        dn = filedialog.askdirectory()
        if dn: entry.delete(0, tk.END); entry.insert(0, dn)

    def _toggle(self):
        spin = (self.mode.get() == "spin")
        for w in (self.b_lbl, self.b_ent, self.b_chk, self.rpm_lbl, self.rpm_e):
            w.grid() if spin else w.grid_remove()
        self.fps_e.configure(state=('disabled' if spin else 'normal'))

    def _run(self):
        # Parse scale
        scale_ratio = None
        scale_in_raw = None
        scale_px_raw = None
        try:
            sin_txt = (self.scale_in.get() or "").strip()
            spx_txt = (self.scale_px.get() or "").strip()
            if sin_txt and spx_txt:
                sin = float(sin_txt); spx = float(spx_txt)
                if spx <= 0: return messagebox.showerror("Invalid input", "Pixels must be > 0.")
                if sin < 0: return messagebox.showerror("Invalid input", "Inches must be ≥ 0.")
                scale_ratio = sin / spx   # inches per pixel
                scale_in_raw = sin
                scale_px_raw = spx
        except ValueError:
            return messagebox.showerror("Invalid input", "Scale fields must be numeric (e.g., 1.0 per 100).")

        # Parse min/max diameter in px
        try:
            min_diam_px = float((self.min_diam_px.get() or "0").strip())
            max_diam_px = float((self.max_diam_px.get() or "0").strip())
            if min_diam_px < 0 or max_diam_px <= 0:
                return messagebox.showerror("Invalid input", "Bubble diameters must be > 0.")
            if min_diam_px >= max_diam_px:
                return messagebox.showerror("Invalid input", "Min diameter must be < max diameter.")
        except ValueError:
            return messagebox.showerror("Invalid input", "Bubble diameters must be numeric.")

        color_name = self.color_var.get()
        overlay_color = COLOR_MAP.get(color_name, COLOR_MAP['Purple'])

        input_type = (self.input_type.get() or "video").strip().lower()
        video_path = (self.vid.get() or "").strip()
        img_dir    = (self.img_dir.get() or "").strip()

        # Default prefix from selected input if empty
        prefix_text = (self.prefix.get() or "").strip()
        if not prefix_text:
            if input_type == "video" and video_path:
                prefix_text = os.path.splitext(os.path.basename(video_path))[0]
            elif input_type == "tif" and img_dir:
                prefix_text = os.path.basename(os.path.normpath(img_dir))
            else:
                prefix_text = "output"
            self.prefix.delete(0, tk.END); self.prefix.insert(0, prefix_text)


        try:
            C = dict(
                INPUT_TYPE=input_type, VIDEO=video_path, IMG_DIR=img_dir, MASK_W=self.mask_w.get(), CNN_W=self.cnn_w.get(),
                OUT_DIR=self.out_dir.get(), PREFIX=prefix_text,
                FPS_STILL=float(self.fps_e.get()), MAX_D=float(self.max_d.get()),
                MAX_MIS=int(self.max_m.get()), IGN_H=int(self.ign_h.get()),
                MIN_DIA_PX=min_diam_px, MAX_DIA_PX=max_diam_px,
                MODE=self.mode.get(), BURST=int(self.b_ent.get()), RESET=self.r_chk.get(),
                GAP_THRESH_MS=DEFAULT_GAP_THRESH_MS, ABSOLUTE_CAP_MS=DEFAULT_ABSOLUTE_CAP_MS,
                IN_BURST_THRESH_MS=DEFAULT_IN_BURST_THRESH_MS,
                SCALE_IN_PER_PX=scale_ratio,
                SCALE_IN_RAW=scale_in_raw,
                SCALE_PX_RAW=scale_px_raw,
                COLOR_NAME=color_name, COLOR_BGR=overlay_color,
                LIQ=self.liq_var.get(), GAS=self.gas_var.get(),
                RPM=float(self.rpm_e.get() or 0.0), RADIUS_IN=RADIUS_IN,
                G_C=G_C, G0=G0_IN_S2,
                SAVE_DEBUG=self.save_debug.get(),
                SAVE_CSVS=self.save_csvs.get(),
            )

        except Exception as e:
            return messagebox.showerror("Invalid input", str(e))

        # Weights must exist
        for k in ("MASK_W", "CNN_W"):
            if not os.path.isfile(C[k]):
                return messagebox.showerror("Error", f"{k} not found.")

        # Input must exist
        if C["INPUT_TYPE"] == "video":
            if not os.path.isfile(C["VIDEO"]):
                return messagebox.showerror("Error", "VIDEO not found.")
        else:
            if not os.path.isdir(C["IMG_DIR"]):
                return messagebox.showerror("Error", "TIFF folder not found.")
            tif_files = list_tif_files(C["IMG_DIR"])
            if len(tif_files) == 0:
                return messagebox.showerror("Error", "No .tif/.tiff files found in that folder.")

        if not os.path.isdir(C['OUT_DIR']): return messagebox.showerror("Error","Output folder missing.")

        fps0 = C['FPS_STILL']; self.prog['value'] = 0
        rows = []
        debug_rows = []

        # --- open debug log for this run ---
        global DEBUG_LOG
        if C['SAVE_DEBUG']:
            log_path = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_debug_log.txt")
            DEBUG_LOG = open(log_path, "w", encoding="utf-8")
        else:
            DEBUG_LOG = None

        try:
            self._track(C, rows, debug_rows, fps0)
        finally:
            if DEBUG_LOG is not None:
                DEBUG_LOG.close()
                DEBUG_LOG = None


        parts = ["✅ Export complete:"]
        parts.append("filtered video")
        if C['SAVE_DEBUG']:
            parts.append("debug videos + debug_pairs.csv")
        if C['SAVE_CSVS']:
            parts.append("tracked.csv + averages.csv")
        msg = " | ".join(parts)
        if C['SCALE_IN_PER_PX']:
            msg += f"\nScale: {C['SCALE_IN_PER_PX']:.6f} in/px"
        msg += f"\nOverlay color: {C['COLOR_NAME']}"
        messagebox.showinfo("Done", msg)

    def _track(self, C, rows, debug_rows, fps0):
    # ---- Phase 1: initialize models, device, and analysis parameters ----
        setup_logger()
                # -------- Detector config: matches training config exactly --------
        cfg = get_cfg()

        # Automatically load config.yaml from same folder as model_final.pth
        cfg_path = os.path.join(os.path.dirname(C['MASK_W']), "config.yaml")
        if not os.path.isfile(cfg_path):
            raise FileNotFoundError(f"config.yaml not found next to weights at:\n{cfg_path}")

        cfg.merge_from_file(cfg_path)

        # Load trained weights
        cfg.MODEL.WEIGHTS = C['MASK_W']
        
        # Predictor threshold controls which instances Detectron2 returns;
        # additional downstream thresholds (DET_SCORE_THRESH and strong/weak split) are applied later.
        # Use same threshold as training visualizer (0.5)
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

        detector = DefaultPredictor(cfg)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Prefer safe weights-only load with allowlist; fall back to classic load
        try:
            from torchvision.models.resnet import ResNet
            import torch.serialization as tser
            tser.add_safe_globals([ResNet])
            classifier = torch.load(C['CNN_W'], map_location=device, weights_only=True)
        except Exception:
            warnings.filterwarnings("ignore", category=FutureWarning, module=r"(torch\.serialization|fvcore\.common\.checkpoint)")
            classifier = torch.load(C['CNN_W'], map_location=device)
        classifier.to(device).eval()

        # fluids (used later for CSV physics)
        liq     = IQ_PROPS[C['LIQ']]
        rho_l   = float(liq['rho'])
        mu_l    = float(liq['mu'])
        sigma   = float(liq['sigma'])
        rho_g   = float(GAS_RHO_FT3[C['GAS']]) / 1728.0
        delta_r = rho_l - rho_g

        if C['MODE'] == 'spin' and C['RPM'] > 0:
            omega = 2*np.pi * C['RPM'] / 60.0
            g_eff = (omega**2) * C['RADIUS_IN']
        else:
            g_eff = C['G0']

        # ---- Phase 2: open input source (video OR tif sequence) ----
        api_pref = cv2.CAP_FFMPEG if hasattr(cv2, "CAP_FFMPEG") else cv2.CAP_ANY

        tif_files = None
        cap = None

        if C["INPUT_TYPE"] == "tif":
            tif_files = list_tif_files(C["IMG_DIR"])
            if len(tif_files) == 0:
                return messagebox.showerror("Input error", "No .tif/.tiff files found in the selected folder.")

            # probe first image
            probe = cv2.imread(tif_files[0], cv2.IMREAD_COLOR)
            if probe is None:
                return messagebox.showerror("Input error", f"Could not read first tif:\n{tif_files[0]}")

            tot = len(tif_files)
            H, W = probe.shape[:2]

            cap_fps = float(C["FPS_STILL"])

            # define a frame reader function
            def read_frame(fi):
                if fi >= tot:
                    return False, None
                img = cv2.imread(tif_files[fi], cv2.IMREAD_COLOR)
                if img is None:
                    return False, None
                # enforce consistent size (optional hard fail if mismatch)
                if img.shape[0] != H or img.shape[1] != W:
                    return False, None
                return True, img

        else:
            cap = cv2.VideoCapture(C["VIDEO"], api_pref)
            if not cap.isOpened():
                return messagebox.showerror("Video error", f"Could not open:\n{C['VIDEO']}")

            ok_probe, probe = cap.read()
            if not ok_probe or probe is None:
                cap.release()
                return messagebox.showerror("Video error", "Opened but cannot read frames. Re-encode to H.264 MP4.")

            # rewind to frame 0
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

            tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or probe.shape[1]
            H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or probe.shape[0]
            src_fps = cap.get(cv2.CAP_PROP_FPS)
            cap_fps = src_fps if src_fps and np.isfinite(src_fps) and src_fps > 1e-3 else 15.0

            def read_frame(fi_unused=None):
                ok, fr = cap.read()
                return ok, fr


        # --- Separate FPS concepts ---
        fps_phys  = max(1e-6, float(C['FPS_STILL']))   # used for time/velocity in STILL mode
        fps_write = float(OUTPUT_FPS)                  # always write output videos at 30 fps

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        p3 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_filtered.mp4")
        vw3 = cv2.VideoWriter(p3, fourcc, fps_write, (W,H))

        # --- binary mask video writer (for IoU) ---
        p_mask = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_pred_mask.mp4")
        vw_mask = cv2.VideoWriter(p_mask, fourcc, fps_write, (W, H))  # needs 3-channel frames
        if not vw_mask.isOpened():
            if cap is not None:
                cap.release()
            vw3.release()
            return messagebox.showerror("Writer error", "Could not open mask video writer.")

        # Debug writers only if requested
        vw1 = vw2 = vw4 = None
        if C['SAVE_DEBUG']:
            p1 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_tracked.mp4")
            p2 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_tracked_cnn.mp4")
            p4 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_debug.mp4")
            vw1 = cv2.VideoWriter(p1, fourcc, fps_write, (W,H))
            vw2 = cv2.VideoWriter(p2, fourcc, fps_write, (W,H))
            vw4 = cv2.VideoWriter(p4, fourcc, fps_write, (W,H))

        if not vw3.isOpened():
            cap.release()
            if vw1: vw1.release()
            if vw2: vw2.release()
            if vw4: vw4.release()
            return messagebox.showerror("Writer error", "Could not open filtered video writer.")

        self.prog['maximum'] = max(1, tot)

        fi, nxt = 0, 0
        tracks = []
        bar_ref_x = None

        # ---------- ONE SOURCE OF TRUTH FOR DISPLAY IDS ----------
        filtered_id_map = {}
        filtered_next_id = 1
        _used_display_ids = set()

        def _alloc_display_id():
            nonlocal filtered_next_id
            while filtered_next_id in _used_display_ids:
                filtered_next_id += 1
            nid = filtered_next_id
            filtered_next_id += 1
            _used_display_ids.add(nid)
            return nid

        def get_display_id(raw_tid: int) -> int:
            if raw_tid not in filtered_id_map:
                filtered_id_map[raw_tid] = _alloc_display_id()
            return filtered_id_map[raw_tid]

        def _assert_id_consistency():
            assert len(set(filtered_id_map.values())) == len(filtered_id_map), "Display ID collision detected!"
        # ---------------------------------------------------------

        dts, times = [], []
        overlay_color = tuple(C['COLOR_BGR'])
        prev_ts = None
        timestamp_debug_rows = []
        

        def track_quality(tr):
            return (2.0*tr.hit_streak + 1.0*tr.cnn_p + 0.3*float(tr.diam_sm)
                    - 0.8*tr.time_since_update - (1.0 if tr.occluded else 0.0))

        # ---- Kalman track class (inside _track)
        class Trk:
            def __init__(s, tid, cen, msk, dia, p):
                s.id = tid
                s.mask = msk.copy()
                s.diam = dia
                s.diam_sm = max(1e-6, dia)
                s.cnn_p = p
                s.kf = KalmanFilter(dim_x=4, dim_z=2)
                s.kf.F = np.eye(4)  # dt injected each predict
                s.kf.H = np.array([[1,0,0,0],[0,1,0,0]], float)
                s.kf.P = np.diag([50.0, 50.0, 2000.0, 2000.0])
                s.sigma_a = 160.0
                s.kf.R = np.diag([ (0.15*s.diam_sm)**2, (0.15*s.diam_sm)**2 ])
                s.kf.x[:2,0] = np.asarray(cen, float)
                s.kf.x[2:,0] = 0.0
                s.miss = 0
                s.prev_meas = np.array(cen, float)
                s.meas = np.array(cen, float)
                s.hit_streak = 1
                s.time_since_update = 0
                s.confirmed = False
                s.occluded = False
                s.occluded_age = 0
                s.merge_into = None
                s.parent_id = None
                s.merged_from_ids = None
                s.split_root_id   = None
                s.label_suffix    = ''
                s.child_ids = set()
                s.merge_streak = 0
                s.valid_visual = (p >= CLS_STAY)

            def _update_Q(s, dt):
                dt2 = dt*dt; dt3 = dt2*dt; dt4 = dt2*dt2
                q = s.sigma_a**2
                s.kf.Q = q * np.array([
                    [dt4/4,   0.0,   dt3/2,  0.0],
                    [0.0,   dt4/4,   0.0,   dt3/2],
                    [dt3/2,  0.0,    dt2,    0.0],
                    [0.0,   dt3/2,   0.0,    dt2]
                ])

            def pred(s, dt):
                s.kf.F[0,2] = dt
                s.kf.F[1,3] = dt
                s._update_Q(dt)
                s.kf.predict()
                s.miss += 1
                s.time_since_update += 1
                s.merged_from_ids = None

            def upd(s, cen, msk, dia, p, dt):
                s.prev_meas = s.meas.copy()
                s.meas = np.array(cen, float)
                chosen = choose_mask(s.mask, msk)
                s.mask = chosen
                area_px = float(np.count_nonzero(chosen))
                s.diam = 2.0 * np.sqrt(area_px / pi) if area_px > 0 else dia
                s.diam_sm = (1.0-EMA_D_ALPHA)*s.diam_sm + EMA_D_ALPHA*max(1e-6, s.diam)
                s.cnn_p = p
                s.valid_visual = (p >= CLS_STAY)
                s.miss = 0
                s.time_since_update = 0
                s.hit_streak += 1
                if not s.confirmed and s.hit_streak >= CONFIRM_HITS:
                    s.confirmed = True
                s.kf.R = np.diag([(0.16*s.diam_sm)**2, (0.16*s.diam_sm)**2])
                z = np.asarray(cen, dtype=float).reshape(2)
                s.kf.update(z)

            def state(s):  # [x,y,vx,vy]
                return s.kf.x.flatten()

        def make_label_text(tr):
            base = get_display_id(tr.split_root_id if tr.split_root_id is not None else tr.id)
            if tr.merged_from_ids and len(tr.merged_from_ids) >= 2 and tr.merge_streak >= MERGE_LABEL_MIN_STREAK:
                parts = []
                for rid in tr.merged_from_ids:
                    parts.append(str(get_display_id(rid)))
                parts = sorted(set(parts), key=lambda s: int(''.join(ch for ch in s if ch.isdigit())))[:2]
                if len(parts) >= 2:
                    return "ID " + "+".join(parts)
            if tr.label_suffix:
                return f"ID {base}{tr.label_suffix}"
            return f"ID {base}"

        # ---- Phase 3: per-frame detection, filtering, association, and track management ----
        while True:
            ok, frame = read_frame(fi)
            if not ok or frame is None:
                break

            current_file = ""
            if C.get("INPUT_TYPE") == "tif" and tif_files is not None and fi < len(tif_files):
                current_file = os.path.basename(tif_files[fi])

            raw_frame = frame.copy()
            # --- timing: physics for STILL, timestamps for SPIN ---
            if C['MODE'] == 'still':
                dt = 1.0 / fps_phys
                time_now = fi / fps_phys
            else:
                # spin mode will overwrite dt/time_now using timestamp later
                dt = 1.0 / max(1e-6, float(fps0))
                time_now = fi / max(1e-6, float(fps0))


            def ahead_xy(tr):
                x, y, vx, vy = tr.state()
                return x, y

            # Spin mode timing
            if C['MODE'] == 'spin':
                bidx, ib = divmod(fi, C['BURST'])
                if C['RESET'] and ib == 0:
                    tracks.clear()
                ts = read_timestamp_tpl(raw_frame)
                ts_cand = float(ts) if ts is not None else None
                
                if ts_cand is not None:
                    # There is timestamp from the frame therefore use it directly
                    if prev_ts is None:
                        prev_ts = ts_cand
                        dt_ms = 0.0             # first frame
                    else:
                        dt_ms = ts_cand - prev_ts
                        if dt_ms < 0:
                            dt_ms = 0.0
                        prev_ts = ts_cand
                else:
                    # OCR completely failed use minimal fallback to keep time moving
                    if dts:
                        dt_ms = dts[-1]
                        prev_ts += dt_ms
                    else:
                        dt_ms = 1000.0 / fps0
                        prev_ts = dt_ms
                
                dts.append(dt_ms)
                times.append(prev_ts)
                dt = max(1e-6, dt_ms / 1000.0)
                time_now = prev_ts / 1000.0

                timestamp_debug_rows.append({
                    "frame": fi,
                    "file": current_file,
                    "raw_ocr_timestamp": ts,
                    "ts_cand_ms": ts_cand,
                    "prev_ts_ms_after_update": prev_ts,
                    "dt_ms": dt_ms,
                    "time_now_s": time_now,
                    "ocr_failed": ts_cand is None,
                })

                do_flip = (bidx % 2 == 1)

                if do_flip:
                    frame = cv2.flip(frame, 1)
                
                dprint(f"Frame {fi}: BURST={C['BURST']}, bidx={bidx}, ib={ib}, do_flip={do_flip}")

                #cv2.putText(
                #    frame,
                #    f"fi={fi} burst={C['BURST']} bidx={bidx} flip={do_flip}",
                #    (20, 40),
                #    cv2.FONT_HERSHEY_SIMPLEX,
                #    0.7,
                #    (0, 255, 255),
                #    2
                #)

            bar_x = find_center_bar_midpoint(frame)

            if fi < 20:
                debug_frame = frame.copy()
            
                H, W = debug_frame.shape[:2]
                dbg_y0 = int(0.02 * H)
                dbg_y1 = int(0.10 * H)
                dbg_x0 = int(0.30 * W)
                dbg_x1 = int(0.70 * W)
            
                cv2.rectangle(debug_frame, (dbg_x0, dbg_y0), (dbg_x1, dbg_y1), (0, 255, 255), 2)
            
                if bar_x is not None:
                    cv2.line(debug_frame, (int(bar_x), 0), (int(bar_x), H), (0, 0, 255), 2)
            
                cv2.imwrite(os.path.join(C['OUT_DIR'], f"rod_debug_frame_{fi:04d}.png"), debug_frame)

            if bar_x is not None:
                if bar_ref_x is None:
                    bar_ref_x = bar_x

                raw_dx = bar_ref_x - bar_x

                # Let the correction change immediately.
                # This is important when BURST=1 because the required correction can alternate every frame.
                max_align_shift_px = 80
                
                if abs(raw_dx) <= max_align_shift_px:
                    dx = raw_dx

                    M = np.float32([[1, 0, dx],
                                    [0, 1, 0]])

                    frame = cv2.warpAffine(
                        frame,
                        M,
                        (W, H),
                        flags=cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_REPLICATE
                    )

                    dprint(
                        f"Frame {fi}: rod_x={bar_x:.2f}, ref={bar_ref_x:.2f}, "
                        f"raw_dx={raw_dx:.2f}, applied_dx={dx:.2f}"
                    )
                else:
                    dprint(
                        f"Frame {fi}: skipped alignment, raw_dx={raw_dx:.2f} too large"
                    )
            else:
                dprint(f"Frame {fi}: rod center not found")

            dprint(f"Frame {fi}: after alignment, bar_ref_x={bar_ref_x}")

            # ------------- Detect --------------
            inst   = detector(frame)['instances'].to('cpu')
            boxes  = inst.pred_boxes.tensor.numpy()
            masks  = inst.pred_masks.numpy()
            scores = inst.scores.numpy()
            if len(scores) > 0:
                dprint(
                    f"RAW Mask-RCNN dets: {len(scores)}  | "
                    f"min score={scores.min():.3f} max={scores.max():.3f}"
                )
            else:
                dprint("RAW Mask-RCNN dets: 0  | no detections this frame")


            dc, dd, dm, dp, ds = [], [], [], [], []
            for bb, m, score in zip(boxes, masks, scores):
                if score < DET_SCORE_THRESH:
                    continue
                m_bool = m.astype(bool)
                M = cv2.moments(m.astype(np.uint8))
                if M['m00'] == 0:
                    continue
                cx = M['m10']/M['m00']; cy = M['m01']/M['m00']
                if cy < C['IGN_H']:
                    continue
                diam = area_to_diam(m_bool)
                if not (C['MIN_DIA_PX'] < diam < C['MAX_DIA_PX']):
                    continue
                roi = crop_center_square(frame, cx, cy)
                x = to_tensor_norm(roi).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits = classifier(x)
                    probs  = torch.softmax(logits, 1)
                    p_bub  = probs[0,0].item()     # adjust index if your model uses 1
                dc.append([cx, cy]); dd.append(diam); dm.append(m_bool); dp.append(p_bub); ds.append(float(score))
            dprint(f"After score/diam/IGN/CNN filter: {len(dc)}")

            # ---- DEBUG: per-detection info BEFORE NMS ----
            dprint(f"---- FRAME {fi} DET DEBUG ----")
            dprint("  RAW Mask-RCNN det count :", len(scores))
            dprint("  Filtered det count      :", len(dc))
            for idx, (cx, cy) in enumerate(dc):
                dprint(f"   DET {idx}: cx={cx:.1f}, cy={cy:.1f}, "
                      f"diam={dd[idx]:.1f}, p_bub={dp[idx]:.2f}, score={ds[idx]:.2f}")
            # ----------------------------------------------
                        
            # Drop near-duplicate detections
            if len(dm) > 1:
                rank = 0.5*np.asarray(ds) + 0.5*np.asarray(dp)
                keep = mask_pairwise_nms(dm, rank, iou_dup=0.82)
                if not np.all(keep):
                    dc = [c for c, k in zip(dc, keep) if k]
                    dd = [d for d, k in zip(dd, keep) if k]
                    dm = [m for m, k in zip(dm, keep) if k]
                    dp = [p for p, k in zip(dp, keep) if k]
                    ds = [s for s, k in zip(ds, keep) if k]

            # ---- DEBUG: after NMS ----
            dprint("  After mask-pairwise NMS:", len(dc))
            for idx, (cx, cy) in enumerate(dc):
                dprint(f"   NMS DET {idx}: cx={cx:.1f}, cy={cy:.1f}")
            # ---------------------------

            dprint(f"Passing into tracker assignment: {len(dc)} detections")
            
            # Split strong / weak 
            strong_idx = [i for i, s in enumerate(ds) if (s >= max(DET_SCORE_THRESH,0.55)) and (dp[i] >= CLS_ENTER)]
            weak_idx   = [i for i, s in enumerate(ds) if (s >= DET_SCORE_THRESH) and not (i in strong_idx)]
            dc_s = [dc[i] for i in strong_idx]
            dd_s = [dd[i] for i in strong_idx]
            dm_s = [dm[i] for i in strong_idx]
            dp_s = [dp[i] for i in strong_idx]

            # ---- DEBUG: split & track state BEFORE predict ----
            dprint("  Strong det idx:", strong_idx)
            dprint("  Weak det idx  :", weak_idx)
            dprint("  Tracks alive before predict:", len(tracks))
            for i, tr in enumerate(tracks):
                x, y, vx, vy = tr.state()
                dprint(f"   TRK {i}: tid={tr.id}, pred_before=({x:.1f},{y:.1f}), "
                      f"diam={tr.diam_sm:.1f}, miss={tr.miss}, confirmed={tr.confirmed}")
            # ---------------------------------------------------
                        
            dt_pred = dt
            for tr in tracks:
                tr.pred(dt_pred)

            debug_spawned = 0
            debug_reassoc = 0
            debug_killed  = 0
            debug_pairs = []

            # Association + handling
            if tracks and (dc_s or weak_idx):
                matched_trk = {}
                matched_det = {}
                used_trks, used_dets = set(), set()

                # Greedy 1↔1
                if len(tracks) == 1 and len(dc_s) == 1:
                    px, py, *_ = tracks[0].state()
                    cx, cy = dc_s[0]
                    if np.hypot(px - cx, py - cy) <= 2.0 * C['MAX_D']:
                        matched_trk[0] = strong_idx[0]
                        matched_det[strong_idx[0]] = 0
                        used_trks.add(0); used_dets.add(strong_idx[0])

                # Hungarian (normalized)
                if not matched_trk and tracks and dc_s:
                    Tn, Dn = len(tracks), len(dc_s)
                    dist_mat = np.zeros((Tn, Dn), float)
                    iou_mat  = np.zeros((Tn, Dn), float)
                    dia_ok   = np.zeros((Tn, Dn), dtype=bool)
                    for i, tr in enumerate(tracks):
                        px, py, _, _ = tr.state()
                        for j, (cx, cy) in enumerate(dc_s):
                            dist_mat[i, j] = np.hypot(px - cx, py - cy)
                            iou_mat[i, j]  = mask_iou(tr.mask, dm_s[j])
                    
                            if tr.hit_streak < 5:
                                dia_ok[i, j] = True
                            else:
                                dia_ok[i, j] = diameter_jump_ok(tr.diam_sm, dd_s[j])

                    overlapped_by_any_det = [False] * len(tracks)
                    for i in range(len(tracks)):
                        for j in range(Dn):
                            if iou_mat[i, j] >= IOU_MERGE_THRESH:
                                overlapped_by_any_det[i] = True
                                break
                    for i, tr in enumerate(tracks):
                        tr.merge_streak = (tr.merge_streak + 1) if overlapped_by_any_det[i] else 0

                    diam_list = [getattr(tr, 'diam_sm', 16.0) for tr in tracks]
                    char = max(12.0, 0.8 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                    cost = np.full((Tn, Dn), 1e6, float)
                    for i, tr in enumerate(tracks):
                        for j in range(Dn):
                            if not dia_ok[i, j]:
                                continue
                            m2 = mahalanobis2(tr, dc_s[j])
                            if (m2 <= GATE_MAHAL2):
                                cost[i, j] = (dist_mat[i, j] / char) + IOU_WEIGHT_DEFAULT*(1.0 - iou_mat[i, j])
                    ridx, cidx = linear_sum_assignment(cost)
                    for i, j in zip(ridx, cidx):
                        if cost[i, j] < 1e6 and dist_mat[i, j] <= C['MAX_D']:
                            matched_trk[i] = strong_idx[j]
                            matched_det[strong_idx[j]] = i
                            used_trks.add(i); used_dets.add(strong_idx[j])

                # Unmatched sets (all dets)
                unmatched_trk_idx = [i for i in range(len(tracks)) if i not in used_trks]
                unmatched_det_idx = [j for j in range(len(dc)) if j not in used_dets]

                # ---- DEBUG: association summary BEFORE merge/split ----
                dprint("  Matched track -> det mapping:")
                for ti, dj in matched_trk.items():
                    dprint(f"   Track idx {ti} (tid={tracks[ti].id}) <- DET {dj}")
                dprint("  Unmatched track idx:", unmatched_trk_idx)
                dprint("  Unmatched det idx  :", unmatched_det_idx)
                # -------------------------------------------------------

                # MERGE & SPLIT bookkeeping
                if dc:
                    Tn = len(tracks); Dn_all = len(dc)
                    iou_mat_all = np.zeros((Tn, Dn_all), float)
                    for i, tr in enumerate(tracks):
                        for j in range(Dn_all):
                            iou_mat_all[i, j] = mask_iou(tr.mask, dm[j])
                    # MERGE
                    for j in range(Dn_all):
                        overlappers = [i for i in range(Tn) if iou_mat_all[i, j] >= IOU_MERGE_THRESH]
                        if len(overlappers) >= 2:
                            if j in matched_det:
                                primary_i = matched_det[j]
                            else:
                                primary_i = max(overlappers, key=lambda i: iou_mat_all[i, j])
                                matched_trk[primary_i] = j
                                matched_det[j] = primary_i
                                used_trks.add(primary_i); used_dets.add(j)
                                if primary_i in unmatched_trk_idx: unmatched_trk_idx.remove(primary_i)
                                if j in unmatched_det_idx: unmatched_det_idx.remove(j)
                            primary = tracks[primary_i]
                            merged_set = set()
                            for i in overlappers:
                                merged_set.add(tracks[i].id)
                                if i != primary_i:
                                    tr = tracks[i]
                                    tr.occluded = True; tr.occluded_age = 0
                                    tr.merge_into = primary.id
                                    tr.parent_id = primary.id
                                    primary.child_ids.add(tr.id)
                            primary.merged_from_ids = merged_set
                    # SPLIT
                    id_to_index = {tr.id: idx for idx, tr in enumerate(tracks)}
                    for i in range(Tn):
                        overlapped_dets = [j for j in range(Dn_all) if iou_mat_all[i, j] >= IOU_SPLIT_THRESH]
                        if len(overlapped_dets) >= 2:
                            member_idxs = [i]
                            for cid in getattr(tracks[i], 'child_ids', set()):
                                idx = id_to_index.get(cid, None)
                                if idx is not None:
                                    member_idxs.append(idx)
                            seen = set(); member_idxs = [m for m in member_idxs if (m not in seen and not seen.add(m))]
                            diam_list = [getattr(tracks[k], 'diam_sm', 16.0) for k in member_idxs]
                            char_local = max(8.0, 0.5 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                            pairs = assign_to_members(member_idxs, overlapped_dets, dc, dm, tracks,
                                                      iou_weight=IOU_WEIGHT_DEFAULT, char_dist=char_local)
                            for ti, dj in pairs:
                                prev_d = matched_trk.get(ti, None)
                                if prev_d is not None:
                                    px, py, *_ = tracks[ti].state()
                                    pc = np.array([px, py]); c_prev = np.array(dc[prev_d]); c_new = np.array(dc[dj])
                                    prev_cost = np.hypot(*(pc - c_prev)) + IOU_WEIGHT_DEFAULT*(1.0 - mask_iou(tracks[ti].mask, dm[prev_d]))
                                    new_cost  = np.hypot(*(pc - c_new))  + IOU_WEIGHT_DEFAULT*(1.0 - mask_iou(tracks[ti].mask, dm[dj]))
                                    if new_cost >= prev_cost:
                                        continue
                                    used_dets.discard(prev_d)
                                    matched_det.pop(prev_d, None)
                                    if prev_d not in unmatched_det_idx:
                                        unmatched_det_idx.append(prev_d)
                                matched_trk[ti] = dj
                                matched_det[dj] = ti
                                used_trks.add(ti); used_dets.add(dj)
                                if dj in unmatched_det_idx: unmatched_det_idx.remove(dj)
                            if len(pairs) >= 2 and hasattr(tracks[i], 'child_ids'):
                                tracks[i].child_ids.clear()

                # collect debug pair metrics
                if C['SAVE_DEBUG'] and tracks and dc:
                    for ti, dj in matched_trk.items():
                        px, py, *_ = tracks[ti].state()
                        cx, cy = dc[dj]
                        d   = float(np.hypot(px - cx, py - cy))
                        iou = float(mask_iou(tracks[ti].mask, dm[dj]))
                        m2  = float(mahalanobis2(tracks[ti], dc[dj]))
                        debug_pairs.append({'frame': fi, 'track_id': tracks[ti].id, 'det_idx': dj,
                                            'd_px': d, 'iou': iou, 'mahal2': m2,
                                            'time_since_update': tracks[ti].time_since_update})

                # Update matched
                for i, j in matched_trk.items():
                    tr = tracks[i]
                    tr.occluded = False; tr.occluded_age = 0; tr.merge_into = None
                    tr.upd(dc[j], dm[j], dd[j], dp[j], dt_pred)
                dprint(f"Matched {len(matched_trk)} tracks this frame")

                # Re-associate to weak dets
                if weak_idx:
                    ut = [i for i in range(len(tracks)) if i not in used_trks and 0 < tracks[i].time_since_update <= MAX_COAST]
                    if ut:
                        dc_w = [dc[i] for i in weak_idx]; dm_w = [dm[i] for i in weak_idx]; dd_w = [dd[i] for i in weak_idx]
                        M, N = len(ut), len(dc_w)
                        cost_w = np.full((M, N), 1e6, float)
                        diam_list = [getattr(tracks[ti], 'diam_sm', 16.0) for ti in ut]
                        char = max(8.0, 0.5 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                        for a, ti in enumerate(ut):
                            tr = tracks[ti]
                            px, py, *_ = tr.state()
                            for b in range(N):
                                cx, cy = dc_w[b]
                                d = np.hypot(px - cx, py - cy)
                                iou = mask_iou(tr.mask, dm_w[b])
                                m2 = mahalanobis2(tr, dc_w[b])
                                if (m2 <= 2.0*GATE_MAHAL2) and (iou >= 0.02 or d <= 1.5*C['MAX_D']):
                                    cost_w[a, b] = (d / char) + IOU_WEIGHT_DEFAULT*(1.0 - iou)
                        rr, cc = linear_sum_assignment(cost_w)
                        for a, b in zip(rr, cc):
                            if cost_w[a, b] < 1e6 and dp[weak_idx[b]] >= 0.30:
                                ti = ut[a]; dj_global = weak_idx[b]
                                tr = tracks[ti]
                                tr.occluded = False; tr.occluded_age = 0; tr.merge_into = None
                                tr.upd(dc[dj_global], dm[dj_global], dd[dj_global], dp[dj_global], dt_pred)
                                used_trks.add(ti)
                                if dj_global in unmatched_det_idx:
                                    unmatched_det_idx.remove(dj_global)
                                debug_reassoc += 1

                # Re-associate unmatched dets
                if unmatched_det_idx:
                    cand_idxs = [i for i, tr in enumerate(tracks)
                                 if tr.time_since_update > 0 and tr.time_since_update <= MAX_COAST]
                    diam_list = [getattr(tracks[i], 'diam_sm', 16.0) for i in cand_idxs] if cand_idxs else []
                    char = max(8.0, 0.5 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                    for j in list(unmatched_det_idx):
                        if dp[j] < 0.25:
                            continue
                        best_i, best_score = None, 1e9
                        for i in cand_idxs:
                            tr = tracks[i]
                            px, py, *_ = tr.state()
                            cx, cy = dc[j]
                            d = np.hypot(px - cx, py - cy)
                            iou = mask_iou(tr.mask, dm[j])
                            m2  = mahalanobis2(tr, dc[j])
                            if (m2 <= 2.0*GATE_MAHAL2) and (iou >= 0.02 or d <= 1.5*C['MAX_D']):
                                sc = (d / char) + IOU_WEIGHT_DEFAULT * (1.0 - iou)
                                if sc < best_score:
                                    best_score, best_i = sc, i
                        if best_i is not None:
                            tr = tracks[best_i]
                            tr.occluded = False; tr.occluded_age = 0; tr.merge_into = None
                            tr.upd(dc[j], dm[j], dd[j], dp[j], dt_pred)
                            used_trks.add(best_i)
                            unmatched_det_idx.remove(j)
                            debug_reassoc += 1

                # Controlled spawning
                MAX_SPAWN_PER_FRAME = 12
                SPAWN_SUPPRESS_FACTOR = 0.5  # how far from an existing bubble instance must be to allow a new track
                spawns = 0
                for j in unmatched_det_idx:
                    if dp[j] < CLS_ENTER:
                        continue
                    dj_cx, dj_cy = dc[j]
                    too_close = False
                
                    # if there is ANY nearby track, don't spawn a new one
                    for tr in tracks:
                        if tr.time_since_update <= MAX_COAST:  # don't require confirmed; just "alive"
                            cx, cy, *_ = tr.state()
                            d = np.hypot(cx - dj_cx, cy - dj_cy)
                            if d <= SPAWN_SUPPRESS_FACTOR * max(1.0, tr.diam_sm):
                                too_close = True
                                break
                
                    if too_close:
                        continue  # treat as a weird duplicate detection, not a new bubble
                
                    if spawns < MAX_SPAWN_PER_FRAME:
                        tracks.append(Trk(nxt, dc[j], dm[j], dd[j], dp[j]))
                        nxt += 1
                        spawns += 1
                        debug_spawned += 1

                # Age/prune
                alive = []
                for tr in tracks:
                    to_kill = False
                    if not tr.confirmed and (tr.miss > max(2, C['MAX_MIS'])):
                        to_kill = True
                    if tr.confirmed and tr.time_since_update > MAX_COAST:
                        to_kill = True
                    if tr.occluded and tr.occluded_age > MAX_OCCLUDE_AGE:
                        to_kill = True
                    if to_kill:
                        debug_killed += 1
                        continue
                    alive.append(tr)
                tracks = alive

                # Track-level NMS
                def tracks_pairwise_nms(tracks_list, iou_thr=0.86):
                    if len(tracks_list) <= 1:
                        return tracks_list
                    order = sorted(range(len(tracks_list)),
                                   key=lambda i: track_quality(tracks_list[i]),
                                   reverse=True)
                    keep_mask = [True]*len(tracks_list)
                    for a in range(len(order)):
                        i = order[a]
                        if not keep_mask[i]:
                            continue
                        for b in range(a+1, len(order)):
                            j = order[b]
                            if not keep_mask[j]:
                                continue
                            iou = mask_iou(tracks_list[i].mask, tracks_list[j].mask)
                            if iou >= iou_thr:
                                keep_mask[j] = False
                    return [t for t,k in zip(tracks_list, keep_mask) if k]
                tracks = tracks_pairwise_nms(tracks, iou_thr=0.86)

            else:
                # No tracks or no strong dets so spawn from all dets that pass enter
                for j in range(len(dc)):
                    if dp[j] >= CLS_ENTER:
                        tracks.append(Trk(nxt, dc[j], dm[j], dd[j], dp[j])); nxt += 1
                tracks = [tr for tr in tracks if tr.miss <= C['MAX_MIS']]
                debug_spawned = sum(1 for j in range(len(dc)) if dp[j] >= CLS_ENTER)

            # ---- Phase 4: write annotated outputs (filtered + optional debug videos) ----
            alpha = 0.4

            # v1: raw tracker IDs (debug)
            if vw1 is not None:
                v1 = frame.copy()
                for tr in tracks:
                    v1[tr.mask] = (v1[tr.mask]*(1 - alpha) + np.array(overlay_color)*alpha).astype(np.uint8)
                for tr in tracks:
                    ax, ay = ahead_xy(tr)
                    cx, cy = int(round(ax)), int(round(ay))
                    r = int(tr.diam_sm / 2)
                    cv2.circle(v1, (cx, cy), r, overlay_color, 2)
                    cv2.putText(v1, f"ID{tr.id}", (cx + 8, cy - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, overlay_color, 2)
                vw1.write(v1)

            # v2: raw ID + CNN prob (debug)
            if vw2 is not None:
                v2 = frame.copy()
                for tr in tracks:
                    ax, ay = ahead_xy(tr)
                    cx, cy = int(round(ax)), int(round(ay))
                    r = int(tr.diam_sm / 2)
                    col = (0, 255, 0) if tr.cnn_p >= CLS_STAY else (0, 0, 255)
                    cv2.rectangle(v2, (cx - r, cy - r), (cx + r, cy + r), col, 2)
                    cv2.putText(v2, f"ID{tr.id} p={tr.cnn_p:.2f}", (cx + 8, cy - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
                vw2.write(v2)

            # v3: filtered (representatives only) — display IDs
            filtered = frame.copy()
            
            # --- only draw tracks that were actually matched this frame ---
            draw_tracks = [tr for tr in tracks if tr.confirmed 
                                                  and tr.valid_visual 
                                                  and tr.miss == 0         # MUST have a detection this frame
                                                  and not tr.occluded]
            
            # ---- DEBUG: which tracks are actually drawn ----
            dprint("  Draw-track tids this frame:", [tr.id for tr in draw_tracks])
            dprint(f"Drawing {len(draw_tracks)} bubbles this frame")
            
            # --- draw overlays ---
            for tr in draw_tracks:
                filtered[tr.mask] = (filtered[tr.mask]*(1 - alpha) + np.array(overlay_color)*alpha).astype(np.uint8)
            
            for tr in draw_tracks:
                ax, ay = ahead_xy(tr)
                cx, cy = int(round(ax)), int(round(ay))
                r = int(tr.diam_sm / 2)
                cv2.circle(filtered, (cx, cy), r, overlay_color, 2)
                disp = get_display_id(tr.id)
                cv2.putText(filtered, f"ID {disp}", (cx + 8, cy - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, overlay_color, 2)

            if fi < 40:
                final_check = filtered.copy()
                cv2.putText(
                    final_check,
                    f"FINAL: fi={fi} burst={bidx} inburst={ib} flip={do_flip}",
                    (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 255),
                    2
                )
                cv2.imwrite(os.path.join(C['OUT_DIR'], f"FINAL_output_{fi:04d}.png"), final_check)
            
            vw3.write(filtered)


            # v4: DEBUG OVERLAY
            if vw4 is not None:
                v4 = frame.copy()
                # A) detections
                for j, (cx, cy) in enumerate(dc):
                    cv2.circle(v4, (int(cx), int(cy)), 6, (255, 100, 0), 2)
                    cv2.putText(v4, f"D{j}", (int(cx)+7, int(cy)-7), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,100,0), 1)
                # B) tracks prediction + id
                for i, tr in enumerate(tracks):
                    x, y, vx, vy = tr.state()
                    cv2.drawMarker(v4, (int(x), int(y)), (255,255,255),
                                   markerType=cv2.MARKER_TILTED_CROSS, markerSize=14, thickness=2)
                    M = cv2.moments(tr.mask.astype(np.uint8))
                    if M['m00'] > 0:
                        mx, my = int(M['m10']/M['m00']), int(M['m01']/M['m00'])
                        x, y, vx, vy = tr.state()
                        dprint(f"ID {tr.id}: track center=({x:.1f},{y:.1f}), "
                              f"mask centroid=({mx},{my}), diam={tr.diam_sm:.1f}")

                        color = (0,255,0) if tr.confirmed else (0,0,255)
                        cv2.circle(v4, (mx, my), max(2, int(tr.diam_sm/2)), color, 1)
                    tag = f"T{tr.id} hit={tr.hit_streak} miss={tr.miss} tsu={tr.time_since_update}"
                    cv2.putText(v4, tag, (int(x)+8, int(y)+16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)
                # C) matched metrics text (optional)
                hud = [
                    f"frame={fi} dets={len(dc)} tracks={len(tracks)}",
                    f"spawned={debug_spawned} re-assoc={debug_reassoc} killed={debug_killed}",
                    f"MAX_D={C['MAX_D']} GATE_M2={GATE_MAHAL2} IOU_W={IOU_WEIGHT_DEFAULT}"
                ]
                y0 = 24
                for line in hud:
                    cv2.putText(v4, line, (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)
                    y0 += 22
                vw4.write(v4)

            # --- write binary mask frame ---
            mask_u8 = np.zeros((H, W), dtype=np.uint8)
            for tr in draw_tracks:
                mask_u8[tr.mask] = 255  # union mask
            
            # mp4v writer expects 3-channel BGR
            mask_bgr = cv2.cvtColor(mask_u8, cv2.COLOR_GRAY2BGR)
            vw_mask.write(mask_bgr)


            # CSV rows: all confirmed & visually valid tracks (use display id)
            for tr in tracks:
                if tr.confirmed and tr.valid_visual and tr.miss == 0 and not tr.occluded:
                    kx, ky, kvx, kvy = tr.state()
                    mx, my = tr.meas  # raw measurement centroid from detection
            
                    disp_id = get_display_id(tr.id)
                    rows.append({
                        'frame': fi, 'time_s': time_now,
                        'id': disp_id,
            
                        'cx': float(mx),
                        'cy': float(my),
            
                        # keep diam as you already do
                        'diam_px': tr.diam_sm,
                        'cnn_p': tr.cnn_p,
                    })


            if C['SAVE_DEBUG'] and debug_pairs:
                debug_rows.extend(debug_pairs)

            fi += 1; self.prog['value'] = fi; self.update_idletasks()

        # tidy up writers
        if cap is not None:
            cap.release()
        vw3.release()
        if vw1: vw1.release()
        if vw2: vw2.release()
        if vw4: vw4.release()
        if vw_mask: vw_mask.release()


        # ---- Phase 5: export measurement CSVs and compute dimensionless parameters (if scale provided) ----
        if C['SAVE_DEBUG'] and debug_rows:
            pd.DataFrame(debug_rows).to_csv(
                os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_debug_pairs.csv"),
                index=False
            )

        if C['SAVE_DEBUG'] and timestamp_debug_rows:
            pd.DataFrame(timestamp_debug_rows).to_csv(
                os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_timestamp_debug.csv"),
                index=False
            )

        if C['SAVE_CSVS'] and rows:
            df = pd.DataFrame(rows).sort_values(['id', 'time_s'])

            G = df.groupby('id', sort=False)
            
            if C.get("INPUT_TYPE") == "tif":
                # 2-frame displacement to suppress 0, jump, 0, jump quantization
                dt2 = G['time_s'].diff(2)
                df['vel_x_px_s'] = G['cx'].diff(2) / dt2
                df['vel_y_px_s'] = G['cy'].diff(2) / dt2
            else:
                dt1 = G['time_s'].diff(1)
                df['vel_x_px_s'] = G['cx'].diff(1) / dt1
                df['vel_y_px_s'] = G['cy'].diff(1) / dt1
            
            df['vel_px_s'] = np.hypot(df['vel_x_px_s'], df['vel_y_px_s'])
            
            # clean up non-finite
            for c in ['vel_x_px_s','vel_y_px_s','vel_px_s']:
                df[c] = pd.to_numeric(df[c], errors='coerce')
                df.loc[~np.isfinite(df[c]), c] = np.nan


            # Unit scaling + dimensionless numbers (if scale provided)
            scale = C.get('SCALE_IN_PER_PX')
            if scale:
                df['diam_in']    = df['diam_px'] * scale
                df['vel_x_in_s'] = df['vel_x_px_s'] * scale
                df['vel_y_in_s'] = df['vel_y_px_s'] * scale
                df['vel_in_s']   = df['vel_px_s']   * scale

                gc = C['G_C']
                U = df['vel_in_s']
                L = df['diam_in']
                if 'diam_in' in df.columns:
                    df['Re'] = (rho_l * U * L) / (mu_l * gc)
                    df['Eo'] = (delta_r * g_eff * (L**2)) / (sigma * gc)
                    df['We'] = (rho_l * (U**2) * L) / (sigma * gc)
                    df['Fr'] = (U**2) / (g_eff * L)
                    df['Ga'] = (rho_l * L * np.sqrt(g_eff * L)) / (mu_l * gc)
                    df['Mo'] = (g_eff * (mu_l * gc)**4 * (delta_r)) / ((rho_l**2) * (sigma * gc)**3)
                    df['Bo'] = df['Eo']

            # Write main track CSV
            path_trk = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_tracked.csv")
            df.to_csv(path_trk, index=False, na_rep='', float_format='%.15g')

            # Helper to compute numeric average from a CSV column (avoids NaNs/strings)
            def excel_average_from_csv_column(csv_path, colname):
                values = []
                with open(csv_path, newline='', encoding='utf-8') as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        s = (row.get(colname, '') or '').strip()
                        if s == '':
                            continue
                        try:
                            v = float(s)
                        except Exception:
                            continue
                        if math.isfinite(v):
                            values.append(v)
                return float(sum(values)/len(values)) if values else float('nan')

            # Per-ID and overall averages
            df_num = pd.read_csv(path_trk)
            cols_all = [c for c in [
                'vel_x_px_s','vel_y_px_s','vel_px_s','diam_px',
                'vel_x_in_s','vel_y_in_s','vel_in_s','diam_in',
                'Re','Eo','We','Fr','Ga','Mo','Bo'
            ] if c in df_num.columns]

            for c in cols_all:
                df_num[c] = pd.to_numeric(df_num[c], errors='coerce')

            per_id = df_num.groupby('id', sort=True)[cols_all].mean().reset_index()
            per_id = per_id.rename(columns={c: f'avg_{c}' for c in cols_all})

            overall = {'id': 'Averages'}
            for c in cols_all:
                overall[f'avg_{c}'] = excel_average_from_csv_column(path_trk, c)

            summary = pd.concat([pd.DataFrame([overall]), per_id], ignore_index=True)
            summary = summary.reindex(columns=['id'] + [f'avg_{c}' for c in cols_all])

            mdot_col = 'avg_mdot_gas_lbm_s'
            summary[mdot_col] = np.nan 
            
            try:
                # Must have physical diameter for real mass flow
                if 'diam_in' in df_num.columns and df_num['diam_in'].notna().any():
                    # Plane location in pixels
                    plane_y = float(COUNT_PLANE_Y_FRAC) * float(H)

                    # Video duration (seconds) from the actual timestamps
                    t0 = float(df_num['time_s'].min())
                    t1 = float(df_num['time_s'].max())
                    T = max(1e-12, (t1 - t0))

                    # Ensure sorted within each ID
                    df_sorted = df_num.sort_values(['id', 'time_s']).copy()

                    # Previous cy per ID
                    df_sorted['cy_prev'] = df_sorted.groupby('id')['cy'].shift(1)

                    # Define crossing condition
                    if COUNT_DIRECTION.lower() == "up":
                        # rising: cy decreases across plane
                        cross = (df_sorted['cy_prev'] > plane_y) & (df_sorted['cy'] <= plane_y)
                    elif COUNT_DIRECTION.lower() == "down":
                        # falling: cy increases across plane
                        cross = (df_sorted['cy_prev'] < plane_y) & (df_sorted['cy'] >= plane_y)
                    else:
                        # both directions
                        cross_up = (df_sorted['cy_prev'] > plane_y) & (df_sorted['cy'] <= plane_y)
                        cross_dn = (df_sorted['cy_prev'] < plane_y) & (df_sorted['cy'] >= plane_y)
                        cross = cross_up | cross_dn

                    # Keep only rows where a crossing occurs
                    df_cross = df_sorted.loc[cross].copy()

                    if not df_cross.empty:
                        # Count each ID once: take FIRST crossing per ID
                        df_first = df_cross.groupby('id', sort=True).head(1)

                        # Diameter at crossing (fallback to median per ID if missing)
                        med_diam = df_sorted.groupby('id', sort=True)['diam_in'].median()
                        d = df_first['diam_in'].copy()
                        d = d.fillna(df_first['id'].map(med_diam))

                        # Drop any IDs still missing diameter
                        d = d[pd.notna(d) & np.isfinite(d) & (d > 0)]

                        if len(d) > 0:
                            # Sphere volume [in^3]
                            vol_in3 = (math.pi / 6.0) * (d ** 3)

                            # Mass per bubble [lbm]
                            mass_lbm = rho_g * vol_in3

                            # Overall mdot [lbm/s]
                            mdot_overall = float(mass_lbm.sum() / T)

                            summary.loc[0, mdot_col] = mdot_overall

            except Exception:
                # Leave blank instead of breaking exports
                pass

            # Force mdot column to be the LAST column
            cols_now = [c for c in summary.columns if c != mdot_col] + [mdot_col]
            summary = summary.reindex(columns=cols_now)
            
            path_avg = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_averages.csv")

            settings_rows = [
                ['# BubbleTracker GUI Settings'],
                ['VIDEO',                 C['VIDEO']],
                ['MASK_WEIGHTS',          C['MASK_W']],
                ['CNN_WEIGHTS',           C['CNN_W']],
                ['MODE',                  C['MODE']],
                ['FPS_STILL',             C['FPS_STILL']],
                ['CAP_FPS_VIDEO',         cap_fps],                  
                ['MAX_ASSIGN_DIST_PX',    C['MAX_D']],
                ['MAX_MISSES_FRAMES',     C['MAX_MIS']],
                ['IGNORE_TEXT_PX',        C['IGN_H']],
                ['MIN_DIAM_PX',           C['MIN_DIA_PX']],
                ['MAX_DIAM_PX',           C['MAX_DIA_PX']],
                ['SCALE_IN_INPUT',        C.get('SCALE_IN_RAW')],    # inches input from GUI
                ['SCALE_PX_INPUT',        C.get('SCALE_PX_RAW')],    # pixel input from GUI
                ['SCALE_IN_PER_PX',       C.get('SCALE_IN_PER_PX')],
                ['LIQUID',                C['LIQ']],
                ['GAS',                   C['GAS']],
                ['RPM',                   C['RPM']],
                ['RADIUS_IN',             C['RADIUS_IN']],
                ['FRAMES_PER_BURST',      C['BURST']],
                ['RESET_IDS_PER_BURST',   C['RESET']],
                ['SAVE_DEBUG',            C['SAVE_DEBUG']],
                ['SAVE_MEASUREMENT_CSVS', C['SAVE_CSVS']],
            ]
            
            with open(path_avg, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                for row in settings_rows:
                    writer.writerow(row)
                writer.writerow([])  # blank line
            
                summary.to_csv(f, index=False, float_format='%.15g')



# --- end class BubbleTrackerApp ---
if __name__ == '__main__':
    BubbleTrackerApp().mainloop()


Loading config C:/Users/PRC/Documents/BLENDER_Python/Bubble-Evaluation-Tool/Models/Mask RCNN/Water\config.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[05/20 07:52:36 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from C:/Users/PRC/Documents/BLENDER_Python/Bubble-Evaluation-Tool/Models/Mask RCNN/Water/model_final.pth ...


C:\Users\PRC\AppData\Local\Temp\ipykernel_6472\1228988145.py:699: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  classifier = torch.load(C['CNN_W'], map_location=device)


In [1]:
# ============================================================
# Bubble Evaluation Tool (Appendix Code)
# Purpose: Automated bubble detection, tracking, and measurement from CNTR video data
# Inputs: CNTR video + Mask R-CNN weights + CNN weights + analysis settings (scale, fluids, mode)
# Outputs: filtered/annotated video(s), tracked.csv, averages.csv (+ optional debug logs)
# Methods: Mask R-CNN segmentation, CNN verification, Kalman + assignment tracking,
#          post-processing and dimensionless parameter calculations
# Note: Running requires Detectron2, PyTorch, FilterPy, and trained model weights.
# ============================================================

import os
import cv2
import numpy as np
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pandas as pd
import math, csv, warnings
from math import pi

import glob
import re

# Detectron2 imports
from detectron2.config          import get_cfg
from detectron2.engine          import DefaultPredictor
from detectron2.utils.logger    import setup_logger
from detectron2                 import model_zoo

# Tracking imports
from filterpy.kalman            import KalmanFilter
from scipy.optimize             import linear_sum_assignment

# CNN imports
import torch
from torchvision import transforms
from PIL import Image

DEBUG_LOG = None   # global handle that will open per run

def dprint(*args, **kwargs):
    global DEBUG_LOG
    if DEBUG_LOG is not None:
        print(*args, **kwargs, file=DEBUG_LOG)

# ---------------- Fluids / physics ----------------
# Note: Dimensionless numbers are computed using consistent English units with gc (G_C).
IQ_PROPS = {
    "Water": {"mu": 0.0000001412232, "sigma": 0.000416, "rho": 0.03605},   # mu: lbf*s/in^2, sigma: lbf/in, rho: lbm/in^3
    "SPT3" : {"mu": 0.000005076321, "sigma": 0.000406, "rho": 0.10838},
}
LIQ_PROPS = IQ_PROPS
GAS_RHO_FT3 = {"Air": 0.07487, "Nitrogen": 0.07210}  # lbm/ft^3

G_C       = 386.4   # (lbm*in)/(lbf*s^2)
G0_IN_S2  = 386.4   # in/s^2
RADIUS_IN = 3.125   # in

# ---------------- Tunables ----------------
IOU_WEIGHT_DEFAULT   = 3.0   # association balance

# Merge/split bookkeeping
IOU_MERGE_THRESH     = 0.20
IOU_SPLIT_THRESH     = 0.20
IOU_REASSOC_THRESH   = 0.30

# Confirmation & coasting
CONFIRM_HITS         = 3
MAX_COAST            = 14
MAX_OCCLUDE_AGE      = 30

# Viz clustering
MERGE_LABEL_MIN_STREAK = 2
CLUSTER_IOU_THR        = 0.75
CLUSTER_CENTER_FRAC    = 0.55

# Timestamp template match (spin mode)
TEMPLATE_DIR = "templates"
X0, Y0, X1, Y1 = 32, -30, 100, None
MATCH_THRESH = 0.85

digit_templates = {}
for d in "0123456789":
    fn = os.path.join(TEMPLATE_DIR, f"{d}.png")
    tpl = cv2.imread(fn, cv2.IMREAD_GRAYSCALE)
    if tpl is None:
        raise FileNotFoundError(f"Could not load template {fn}")
    _, bw = cv2.threshold(tpl, 127, 255, cv2.THRESH_BINARY_INV)
    digit_templates[d] = bw

def read_timestamp_tpl(frame):
    H, W = frame.shape[:2]
    y1 = H if Y1 is None else Y1
    crop = frame[H+Y0:H+y1, X0:X1]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    res_maps = {d: cv2.matchTemplate(bw, tpl, cv2.TM_CCOEFF_NORMED)
                for d, tpl in digit_templates.items()}
    candidates = []
    w_min = min(tpl.shape[1] for tpl in digit_templates.values())
    Wb = bw.shape[1]
    for x in range(Wb - w_min + 1):
        best_d, best_s = None, -1.0
        for d, res in res_maps.items():
            if x < res.shape[1]:
                s = res[:, x].max()
                if s > best_s:
                    best_s, best_d = s, d
        if best_s >= MATCH_THRESH:
            candidates.append((x, best_d))
    if not candidates:
        return None
    candidates.sort(key=lambda t: t[0])
    out, last_x = [], -999
    for x, d in candidates:
        w = digit_templates[d].shape[1]
        if x - last_x > w // 2:
            out.append((x, d))
            last_x = x
    out.sort(key=lambda t: t[0])
    digits = "".join(d for _, d in out)
    if len(digits) < 4:
        return None
    return digits[:-3] + "." + digits[-3:]

# Colors
COLOR_MAP = {
    "Red":    (0, 0, 255),
    "Orange": (0,165,255),
    "Yellow": (0,255,255),
    "Green":  (0,255,0),
    "Blue":   (255,0,0),
    "Indigo": (130,0,75),
    "Purple": (240,80,160),
    "Pink":   (147,20,255),
}

# -------- detection / gating / smoothing ----------
DET_SCORE_THRESH     = 0.05     # 0.50 
CLS_ENTER            = 0.40     # prob to allow spawn
CLS_STAY             = 0.30     # prob to remain visible/valid

EMA_D_ALPHA          = 0.30
MAX_DIA_GROWTH       = 3.0
MAX_DIA_SHRINK       = 0.40
UPWARD_WEIGHT        = 0.0

GATE_MAHAL2          = 18.0
MIN_IOU_FOR_ASSIGN   = 0.05

ROI_SIZE = 128
DEFAULT_BURST_FRAMES = 8
DEFAULT_GAP_THRESH_MS = 30.0
DEFAULT_ABSOLUTE_CAP_MS = 200.0
DEFAULT_IN_BURST_THRESH_MS = 0.30
OUTPUT_FPS = 30.0

COUNT_PLANE_Y_FRAC = 0.50   # plane at 50% of frame height (0.0 top, 1.0 bottom)
COUNT_DIRECTION    = "up"   # "up" (rising, cy decreases), "down", or "both"


to_tensor_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ----------------------- helpers -----------------------

def mask_iou(a_bool, b_bool):
    inter = np.logical_and(a_bool, b_bool).sum()
    union = np.logical_or(a_bool, b_bool).sum()
    if union == 0:
        return 1.0 if inter == 0 else 0.0
    return float(inter) / float(union)

def mahalanobis2(trk, z):
    H = trk.kf.H
    S = H @ trk.kf.P @ H.T + trk.kf.R
    y = np.array(z, dtype=float).reshape(2,1) - H @ trk.kf.x
    return float((y.T @ np.linalg.inv(S) @ y).ravel())

def choose_mask(prev_mask, new_mask,
                iou_keep_prev=0.25,     
                area_ratio_min=0.55,
                area_ratio_max=1.90):
    prev_area = float(prev_mask.sum())
    new_area  = float(new_mask.sum())
    if prev_area > 0 and new_area > 0:
        iou = mask_iou(prev_mask, new_mask)
        ratio = new_area / prev_area
        if (iou < iou_keep_prev) and (ratio < area_ratio_min or ratio > area_ratio_max):
            return prev_mask
    dprint(f"choose_mask: iou={iou:.3f} ratio={ratio:.2f} -> {'KEEP_PREV' if ((iou < iou_keep_prev) and (ratio < area_ratio_min or ratio > area_ratio_max)) else 'USE_NEW'}")        
    return new_mask

def area_to_diam(mask_bool):
    a = float(mask_bool.sum())
    return 2.0*np.sqrt(a/pi) if a > 0 else 0.0

def diameter_jump_ok(prev_d, new_d):
    if prev_d <= 0 or new_d <= 0:
        return True
    ratio = new_d / prev_d
    return (ratio <= MAX_DIA_GROWTH) and (ratio >= MAX_DIA_SHRINK)

def mask_pairwise_nms(dm, score, iou_dup=0.86):
    n = len(dm)
    keep = np.ones(n, dtype=bool)
    order = np.argsort(-np.asarray(score, float))
    for a in range(n):
        i = order[a]
        if not keep[i]:
            continue
        for b in range(a+1, n):
            j = order[b]
            if not keep[j]:
                continue
            if mask_iou(dm[i], dm[j]) >= iou_dup:
                if score[i] >= score[j]:
                    keep[j] = False
                else:
                    keep[i] = False
                    break
    return keep

def assign_to_members(member_idxs, det_idxs, dc, dm, tracks, iou_weight=IOU_WEIGHT_DEFAULT, char_dist=12.0):
    if not member_idxs or not det_idxs: return []
    M, N = len(member_idxs), len(det_idxs)
    cost = np.full((M, N), 1e6, float)
    for a, ti in enumerate(member_idxs):
        px, py, _, _ = tracks[ti].state()
        for b, dj in enumerate(det_idxs):
            cx, cy = dc[dj]
            d = np.hypot(px - cx, py - cy)
            iou = mask_iou(tracks[ti].mask, dm[dj])
            up_pen = UPWARD_WEIGHT * max(0.0, cy - py)
            cost[a, b] = (d / max(char_dist, 1.0)) + iou_weight*(1.0 - iou) + up_pen
    r, c = linear_sum_assignment(cost)
    return [(member_idxs[i], det_idxs[j]) for i, j in zip(r, c) if cost[i, j] < 1e6]

def crop_center_square(frame, cx, cy, size=ROI_SIZE):
    H, W = frame.shape[:2]
    half = size // 2
    x0, y0 = int(round(cx)) - half, int(round(cy)) - half
    x1, y1 = x0 + size, y0 + size
    patch = np.zeros((size, size, 3), dtype=frame.dtype)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(W, x1), min(H, y1)
    dx0, dy0 = sx0 - x0, sy0 - y0
    dx1, dy1 = dx0 + (sx1 - sx0), dy0 + (sy1 - sy0)
    if sy1 > sy0 and sx1 > sx0 and dy1 > 0:
        patch[dy0:dy1, dx0:dx1] = frame[sy0:sy1, sx0:sx1]
    return Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))

def find_center_bar_midpoint(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape

    # TOP ONLY
    top_y0 = int(0.02 * H)
    top_y1 = int(0.18 * H)

    x0 = int(0.20 * W)
    x1 = int(0.80 * W)

    roi = gray[top_y0:top_y1, x0:x1]
    if roi.size == 0:
        return None

    col_profile = roi.mean(axis=0)

    # assumes the center bar is darker than the background
    thresh = col_profile.min() + 0.5 * (col_profile.max() - col_profile.min())
    cols = np.where(col_profile < thresh)[0]

    if len(cols) == 0:
        return None

    left = cols[0]
    right = cols[-1]
    mid = x0 + 0.5 * (left + right)

    return float(mid)

def cluster_representatives(visible, iou_thr=CLUSTER_IOU_THR, center_frac=CLUSTER_CENTER_FRAC):
    n = len(visible)
    if n <= 1:
        return visible[:]
    adj = [set() for _ in range(n)]
    centers = [np.array(tr.state()[:2], float) for tr in visible]
    diams   = [float(getattr(tr, 'diam_sm', 0.0)) for tr in visible]
    for i in range(n):
        for j in range(i+1, n):
            iou = mask_iou(visible[i].mask, visible[j].mask)
            d   = np.linalg.norm(centers[i] - centers[j])
            close = d <= center_frac * max(1.0, min(diams[i], diams[j]))
            if (iou >= iou_thr) or close:
                adj[i].add(j); adj[j].add(i)
    seen = [False]*n; clusters = []
    for i in range(n):
        if seen[i]: continue
        comp = [i]; seen[i] = True
        stack = [i]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
                    comp.append(v)
        clusters.append(comp)
    reps = []
    for comp in clusters:
        best_tr, best_s = None, -1e9
        for idx in comp:
            tr = visible[idx]
            s = 2.0*tr.hit_streak + 1.0*tr.cnn_p + 0.3*(tr.diam_sm) - 0.8*tr.time_since_update - (1.0 if tr.occluded else 0.0)
            if s > best_s:
                best_s = s; best_tr = tr
        reps.append(best_tr)
    return reps
    
def natural_key(path):
    # natural sort for filenames: Img2 before Img10
    base = os.path.basename(path)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', base)]

def list_tif_files(folder):
    patterns = ["*.tif", "*.tiff", "*.TIF", "*.TIFF"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(folder, pat)))
    files = sorted(files, key=natural_key)
    return files

# ----------------------- GUI app -----------------------
class BubbleTrackerApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Bubble Tracker GUI")
        self._build_ui()

    def _build_ui(self):
        pad, row = {"padx":4, "pady":2}, 0
        def add(lbl, ent):
            nonlocal row
            ttk.Label(self, text=lbl).grid(row=row, column=0, sticky="e", **pad)
            ent.grid(row=row, column=1, **pad)
            row += 1

        # Inputs
        self.vid = ttk.Entry(self, width=42);        add("Input Video:", self.vid)
        ttk.Button(self, text="Browse…", command=lambda: self._browse(self.vid, [("VIDEO","*.avi *.mp4 *.mov *.mkv *.m4v")])).grid(row=row-1, column=2, **pad)

        self.img_dir = ttk.Entry(self, width=42);    add("Input TIFF Folder:", self.img_dir)
        ttk.Button(self, text="Browse…", command=lambda: self._browse_dir(self.img_dir)).grid(row=row-1, column=2, **pad)

        ttk.Label(self, text="Input Type:").grid(row=row, column=0, sticky="e", **pad)
        self.input_type = tk.StringVar(value="video")
        ttk.Radiobutton(self, text="Video", variable=self.input_type, value="video").grid(row=row, column=1, sticky="w", **pad)
        ttk.Radiobutton(self, text="TIFF folder", variable=self.input_type, value="tif").grid(row=row, column=2, sticky="w", **pad)
        row += 1

        self.mask_w = ttk.Entry(self, width=42);     add("Mask-R-CNN Weights:", self.mask_w)
        ttk.Button(self, text="Browse…", command=lambda: self._browse(self.mask_w, [("PyTorch","*.pth *.pt")])).grid(row=row-1, column=2, **pad)

        self.cnn_w = ttk.Entry(self, width=42);      add("CNN Weights:", self.cnn_w)
        ttk.Button(self, text="Browse…", command=lambda: self._browse(self.cnn_w, [("PyTorch","*.pth *.pt")])).grid(row=row-1, column=2, **pad)

        self.out_dir = ttk.Entry(self, width=42);    add("Output Folder:", self.out_dir)
        ttk.Button(self, text="Browse…", command=lambda: self._browse_dir(self.out_dir)).grid(row=row-1, column=2, **pad)

        self.prefix = ttk.Entry(self, width=14); self.prefix.insert(0, ""); add("Filename Prefix:", self.prefix)

        self.fps_e = ttk.Entry(self, width=6); self.fps_e.insert(0, "1200"); add("FPS (still mode):", self.fps_e)
        self.max_d = ttk.Entry(self, width=6); self.max_d.insert(0, "65"); add("Max assign dist (px):", self.max_d)
        self.max_m = ttk.Entry(self, width=6); self.max_m.insert(0, "5");  add("Max misses (frames):", self.max_m)
        self.ign_h = ttk.Entry(self, width=6); self.ign_h.insert(0, "0");  add("Ignore text (px):", self.ign_h)

        # bubble dia
        ttk.Label(self, text="Bubble diameter (px):").grid(row=row, column=0, sticky="e", **pad)
        dframe = ttk.Frame(self)
        self.min_diam_px = ttk.Entry(dframe, width=6); self.min_diam_px.insert(0, "5")
        self.max_diam_px = ttk.Entry(dframe, width=6); self.max_diam_px.insert(0, "240")
        ttk.Label(dframe, text="min").grid(row=0, column=0, **pad)
        self.min_diam_px.grid(row=0, column=1, **pad)
        ttk.Label(dframe, text="max").grid(row=0, column=2, **pad)
        self.max_diam_px.grid(row=0, column=3, **pad)
        dframe.grid(row=row, column=1, columnspan=2, sticky="w", **pad)
        row += 1

        # scale
        ttk.Label(self, text="Scale:").grid(row=row, column=0, sticky="e", **pad)
        scf = ttk.Frame(self)
        self.scale_in = ttk.Entry(scf, width=8); self.scale_px = ttk.Entry(scf, width=8)
        self.scale_in.insert(0, "0.5"); self.scale_px.insert(0, "100.0")
        self.scale_in.grid(row=0, column=0, **pad); ttk.Label(scf, text="Inches per").grid(row=0, column=1, **pad)
        self.scale_px.grid(row=0, column=2, **pad); ttk.Label(scf, text="pixels").grid(row=0, column=3, **pad)
        scf.grid(row=row, column=1, columnspan=2, sticky="w", **pad); row += 1

        # fluids
        ttk.Label(self, text="Liquid:").grid(row=row, column=0, sticky="e", **pad)
        self.liq_var = tk.StringVar(value="Water")
        ttk.Combobox(self, textvariable=self.liq_var,
                     values=list(IQ_PROPS.keys()), state="readonly", width=14
        ).grid(row=row, column=1, columnspan=2, sticky="w", **pad)
        row += 1

        ttk.Label(self, text="Gas:").grid(row=row, column=0, sticky="e", **pad)
        self.gas_var = tk.StringVar(value="Air")
        ttk.Combobox(self, textvariable=self.gas_var,
                     values=list(GAS_RHO_FT3.keys()), state="readonly", width=14
        ).grid(row=row, column=1, columnspan=2, sticky="w", **pad)
        row += 1

        ttk.Label(self, text="Overlay Color:").grid(row=row, column=0, sticky="e", **pad)
        self.color_var = tk.StringVar(value="Purple")
        self.color_cb = ttk.Combobox(self, textvariable=self.color_var, values=list(COLOR_MAP.keys()), state="readonly", width=14)
        self.color_cb.grid(row=row, column=1, columnspan=2, sticky="w", **pad); row += 1

        # mode
        ttk.Label(self, text="Mode:").grid(row=row, column=0, sticky="e", **pad)
        self.mode = tk.StringVar(value="still")
        ttk.Radiobutton(self, text="Still",    variable=self.mode, value="still", command=self._toggle).grid(row=row, column=1, sticky="w")
        ttk.Radiobutton(self, text="Spinning", variable=self.mode, value="spin",  command=self._toggle).grid(row=row, column=2, sticky="w"); row += 1

        self.rpm_lbl = ttk.Label(self, text="RPM (spin):")
        self.rpm_e   = ttk.Entry(self, width=8); self.rpm_e.insert(0, "0")
        self.rpm_lbl.grid(row=row, column=0, sticky="e", **pad)
        self.rpm_e.grid(row=row, column=1, sticky="w", **pad); row += 1

        self.b_lbl = ttk.Label(self, text="Frames/Burst:")
        self.b_ent = ttk.Entry(self, width=6); self.b_ent.insert(0, str(DEFAULT_BURST_FRAMES))
        self.r_chk = tk.BooleanVar(value=True); self.b_chk = ttk.Checkbutton(self, text="Reset IDs per burst", variable=self.r_chk)
        self.b_lbl.grid(row=row, column=0, **pad); self.b_ent.grid(row=row, column=1, **pad); self.b_chk.grid(row=row, column=2, **pad); row += 1

        # output options
        self.save_debug = tk.BooleanVar(value=False)   # v1, v2, v4 + debug_pairs.csv
        self.save_csvs  = tk.BooleanVar(value=True)   # tracked.csv + averages.csv
        ttk.Checkbutton(self, text="Save debug outputs",
                        variable=self.save_debug).grid(row=row, column=0, columnspan=3, sticky="w", padx=4, pady=(8,2))
        row += 1
        ttk.Checkbutton(self, text="Save measurement CSVs",
                        variable=self.save_csvs).grid(row=row, column=0, columnspan=3, sticky="w", padx=4, pady=(0,6))
        row += 1

        ttk.Button(self, text="Run Tracking", command=self._run).grid(row=row, column=0, columnspan=3, pady=(6,4)); row += 1
        self.prog = ttk.Progressbar(self, length=420, mode="determinate"); self.prog.grid(row=row, column=0, columnspan=3, pady=(4,8))
        self._toggle()

    def _browse(self, entry, types):
        fn = filedialog.askopenfilename(filetypes=types)
        if fn:
            entry.delete(0, tk.END); entry.insert(0, fn)
            # Auto-fill prefix with input video basename (no ext) if empty or different video selected
            base = os.path.splitext(os.path.basename(fn))[0]
            cur_prefix = (self.prefix.get() or "").strip()
            if not cur_prefix or cur_prefix == "output":
                self.prefix.delete(0, tk.END)
                self.prefix.insert(0, base)

    def _browse_dir(self, entry):
        dn = filedialog.askdirectory()
        if dn: entry.delete(0, tk.END); entry.insert(0, dn)

    def _toggle(self):
        spin = (self.mode.get() == "spin")
        for w in (self.b_lbl, self.b_ent, self.b_chk, self.rpm_lbl, self.rpm_e):
            w.grid() if spin else w.grid_remove()
        self.fps_e.configure(state=('disabled' if spin else 'normal'))

    def _run(self):
        # Parse scale
        scale_ratio = None
        scale_in_raw = None
        scale_px_raw = None
        try:
            sin_txt = (self.scale_in.get() or "").strip()
            spx_txt = (self.scale_px.get() or "").strip()
            if sin_txt and spx_txt:
                sin = float(sin_txt); spx = float(spx_txt)
                if spx <= 0: return messagebox.showerror("Invalid input", "Pixels must be > 0.")
                if sin < 0: return messagebox.showerror("Invalid input", "Inches must be ≥ 0.")
                scale_ratio = sin / spx   # inches per pixel
                scale_in_raw = sin
                scale_px_raw = spx
        except ValueError:
            return messagebox.showerror("Invalid input", "Scale fields must be numeric (e.g., 1.0 per 100).")

        # Parse min/max diameter in px
        try:
            min_diam_px = float((self.min_diam_px.get() or "0").strip())
            max_diam_px = float((self.max_diam_px.get() or "0").strip())
            if min_diam_px < 0 or max_diam_px <= 0:
                return messagebox.showerror("Invalid input", "Bubble diameters must be > 0.")
            if min_diam_px >= max_diam_px:
                return messagebox.showerror("Invalid input", "Min diameter must be < max diameter.")
        except ValueError:
            return messagebox.showerror("Invalid input", "Bubble diameters must be numeric.")

        color_name = self.color_var.get()
        overlay_color = COLOR_MAP.get(color_name, COLOR_MAP['Purple'])

        input_type = (self.input_type.get() or "video").strip().lower()
        video_path = (self.vid.get() or "").strip()
        img_dir    = (self.img_dir.get() or "").strip()

        # Default prefix from selected input if empty
        prefix_text = (self.prefix.get() or "").strip()
        if not prefix_text:
            if input_type == "video" and video_path:
                prefix_text = os.path.splitext(os.path.basename(video_path))[0]
            elif input_type == "tif" and img_dir:
                prefix_text = os.path.basename(os.path.normpath(img_dir))
            else:
                prefix_text = "output"
            self.prefix.delete(0, tk.END); self.prefix.insert(0, prefix_text)


        try:
            C = dict(
                INPUT_TYPE=input_type, VIDEO=video_path, IMG_DIR=img_dir, MASK_W=self.mask_w.get(), CNN_W=self.cnn_w.get(),
                OUT_DIR=self.out_dir.get(), PREFIX=prefix_text,
                FPS_STILL=float(self.fps_e.get()), MAX_D=float(self.max_d.get()),
                MAX_MIS=int(self.max_m.get()), IGN_H=int(self.ign_h.get()),
                MIN_DIA_PX=min_diam_px, MAX_DIA_PX=max_diam_px,
                MODE=self.mode.get(), BURST=int(self.b_ent.get()), RESET=self.r_chk.get(),
                GAP_THRESH_MS=DEFAULT_GAP_THRESH_MS, ABSOLUTE_CAP_MS=DEFAULT_ABSOLUTE_CAP_MS,
                IN_BURST_THRESH_MS=DEFAULT_IN_BURST_THRESH_MS,
                SCALE_IN_PER_PX=scale_ratio,
                SCALE_IN_RAW=scale_in_raw,
                SCALE_PX_RAW=scale_px_raw,
                COLOR_NAME=color_name, COLOR_BGR=overlay_color,
                LIQ=self.liq_var.get(), GAS=self.gas_var.get(),
                RPM=float(self.rpm_e.get() or 0.0), RADIUS_IN=RADIUS_IN,
                G_C=G_C, G0=G0_IN_S2,
                SAVE_DEBUG=self.save_debug.get(),
                SAVE_CSVS=self.save_csvs.get(),
            )

        except Exception as e:
            return messagebox.showerror("Invalid input", str(e))

        # Weights must exist
        for k in ("MASK_W", "CNN_W"):
            if not os.path.isfile(C[k]):
                return messagebox.showerror("Error", f"{k} not found.")

        # Input must exist
        if C["INPUT_TYPE"] == "video":
            if not os.path.isfile(C["VIDEO"]):
                return messagebox.showerror("Error", "VIDEO not found.")
        else:
            if not os.path.isdir(C["IMG_DIR"]):
                return messagebox.showerror("Error", "TIFF folder not found.")
            tif_files = list_tif_files(C["IMG_DIR"])
            if len(tif_files) == 0:
                return messagebox.showerror("Error", "No .tif/.tiff files found in that folder.")

        if not os.path.isdir(C['OUT_DIR']): return messagebox.showerror("Error","Output folder missing.")

        fps0 = C['FPS_STILL']; self.prog['value'] = 0
        rows = []
        debug_rows = []

        # --- open debug log for this run ---
        global DEBUG_LOG
        if C['SAVE_DEBUG']:
            log_path = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_debug_log.txt")
            DEBUG_LOG = open(log_path, "w", encoding="utf-8")
        else:
            DEBUG_LOG = None

        try:
            self._track(C, rows, debug_rows, fps0)
        finally:
            if DEBUG_LOG is not None:
                DEBUG_LOG.close()
                DEBUG_LOG = None


        parts = ["✅ Export complete:"]
        parts.append("filtered video")
        if C['SAVE_DEBUG']:
            parts.append("debug videos + debug_pairs.csv")
        if C['SAVE_CSVS']:
            parts.append("tracked.csv + averages.csv")
        msg = " | ".join(parts)
        if C['SCALE_IN_PER_PX']:
            msg += f"\nScale: {C['SCALE_IN_PER_PX']:.6f} in/px"
        msg += f"\nOverlay color: {C['COLOR_NAME']}"
        messagebox.showinfo("Done", msg)

    def _track(self, C, rows, debug_rows, fps0):
    # ---- Phase 1: initialize models, device, and analysis parameters ----
        setup_logger()
                # -------- Detector config: matches training config exactly --------
        cfg = get_cfg()

        # Automatically load config.yaml from same folder as model_final.pth
        cfg_path = os.path.join(os.path.dirname(C['MASK_W']), "config.yaml")
        if not os.path.isfile(cfg_path):
            raise FileNotFoundError(f"config.yaml not found next to weights at:\n{cfg_path}")

        cfg.merge_from_file(cfg_path)

        # Load trained weights
        cfg.MODEL.WEIGHTS = C['MASK_W']
        
        # Predictor threshold controls which instances Detectron2 returns;
        # additional downstream thresholds (DET_SCORE_THRESH and strong/weak split) are applied later.
        # Use same threshold as training visualizer (0.5)
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

        detector = DefaultPredictor(cfg)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Prefer safe weights-only load with allowlist; fall back to classic load
        try:
            from torchvision.models.resnet import ResNet
            import torch.serialization as tser
            tser.add_safe_globals([ResNet])
            classifier = torch.load(C['CNN_W'], map_location=device, weights_only=True)
        except Exception:
            warnings.filterwarnings("ignore", category=FutureWarning, module=r"(torch\.serialization|fvcore\.common\.checkpoint)")
            classifier = torch.load(C['CNN_W'], map_location=device)
        classifier.to(device).eval()

        # fluids (used later for CSV physics)
        liq     = IQ_PROPS[C['LIQ']]
        rho_l   = float(liq['rho'])
        mu_l    = float(liq['mu'])
        sigma   = float(liq['sigma'])
        rho_g   = float(GAS_RHO_FT3[C['GAS']]) / 1728.0
        delta_r = rho_l - rho_g

        if C['MODE'] == 'spin' and C['RPM'] > 0:
            omega = 2*np.pi * C['RPM'] / 60.0
            g_eff = (omega**2) * C['RADIUS_IN']
        else:
            g_eff = C['G0']

        # ---- Phase 2: open input source (video OR tif sequence) ----
        api_pref = cv2.CAP_FFMPEG if hasattr(cv2, "CAP_FFMPEG") else cv2.CAP_ANY

        tif_files = None
        cap = None

        if C["INPUT_TYPE"] == "tif":
            tif_files = list_tif_files(C["IMG_DIR"])
            if len(tif_files) == 0:
                return messagebox.showerror("Input error", "No .tif/.tiff files found in the selected folder.")

            # probe first image
            probe = cv2.imread(tif_files[0], cv2.IMREAD_COLOR)
            if probe is None:
                return messagebox.showerror("Input error", f"Could not read first tif:\n{tif_files[0]}")

            tot = len(tif_files)
            H, W = probe.shape[:2]

            cap_fps = float(C["FPS_STILL"])

            # define a frame reader function
            def read_frame(fi):
                if fi >= tot:
                    return False, None
                img = cv2.imread(tif_files[fi], cv2.IMREAD_COLOR)
                if img is None:
                    return False, None
                # enforce consistent size (optional hard fail if mismatch)
                if img.shape[0] != H or img.shape[1] != W:
                    return False, None
                return True, img

        else:
            cap = cv2.VideoCapture(C["VIDEO"], api_pref)
            if not cap.isOpened():
                return messagebox.showerror("Video error", f"Could not open:\n{C['VIDEO']}")

            ok_probe, probe = cap.read()
            if not ok_probe or probe is None:
                cap.release()
                return messagebox.showerror("Video error", "Opened but cannot read frames. Re-encode to H.264 MP4.")

            # rewind to frame 0
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

            tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or probe.shape[1]
            H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or probe.shape[0]
            src_fps = cap.get(cv2.CAP_PROP_FPS)
            cap_fps = src_fps if src_fps and np.isfinite(src_fps) and src_fps > 1e-3 else 15.0

            def read_frame(fi_unused=None):
                ok, fr = cap.read()
                return ok, fr


        # --- Separate FPS concepts ---
        fps_phys  = max(1e-6, float(C['FPS_STILL']))   # used for time/velocity in STILL mode
        fps_write = float(OUTPUT_FPS)                  # always write output videos at 30 fps

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        p3 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_filtered.mp4")
        vw3 = cv2.VideoWriter(p3, fourcc, fps_write, (W,H))

        # --- binary mask video writer (for IoU) ---
        p_mask = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_pred_mask.mp4")
        vw_mask = cv2.VideoWriter(p_mask, fourcc, fps_write, (W, H))  # needs 3-channel frames
        if not vw_mask.isOpened():
            if cap is not None:
                cap.release()
            vw3.release()
            return messagebox.showerror("Writer error", "Could not open mask video writer.")

        # Debug writers only if requested
        vw1 = vw2 = vw4 = None
        if C['SAVE_DEBUG']:
            p1 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_tracked.mp4")
            p2 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_tracked_cnn.mp4")
            p4 = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_debug.mp4")
            vw1 = cv2.VideoWriter(p1, fourcc, fps_write, (W,H))
            vw2 = cv2.VideoWriter(p2, fourcc, fps_write, (W,H))
            vw4 = cv2.VideoWriter(p4, fourcc, fps_write, (W,H))

        if not vw3.isOpened():
            cap.release()
            if vw1: vw1.release()
            if vw2: vw2.release()
            if vw4: vw4.release()
            return messagebox.showerror("Writer error", "Could not open filtered video writer.")

        self.prog['maximum'] = max(1, tot)

        fi, nxt = 0, 0
        tracks = []
        bar_ref_x = None

        # ---------- ONE SOURCE OF TRUTH FOR DISPLAY IDS ----------
        filtered_id_map = {}
        filtered_next_id = 1
        _used_display_ids = set()

        def _alloc_display_id():
            nonlocal filtered_next_id
            while filtered_next_id in _used_display_ids:
                filtered_next_id += 1
            nid = filtered_next_id
            filtered_next_id += 1
            _used_display_ids.add(nid)
            return nid

        def get_display_id(raw_tid: int) -> int:
            if raw_tid not in filtered_id_map:
                filtered_id_map[raw_tid] = _alloc_display_id()
            return filtered_id_map[raw_tid]

        def _assert_id_consistency():
            assert len(set(filtered_id_map.values())) == len(filtered_id_map), "Display ID collision detected!"
        # ---------------------------------------------------------

        dts, times = [], []
        overlay_color = tuple(C['COLOR_BGR'])
        prev_ts = None

        def track_quality(tr):
            return (2.0*tr.hit_streak + 1.0*tr.cnn_p + 0.3*float(tr.diam_sm)
                    - 0.8*tr.time_since_update - (1.0 if tr.occluded else 0.0))

        # ---- Kalman track class (inside _track)
        class Trk:
            def __init__(s, tid, cen, msk, dia, p):
                s.id = tid
                s.mask = msk.copy()
                s.diam = dia
                s.diam_sm = max(1e-6, dia)
                s.cnn_p = p
                s.kf = KalmanFilter(dim_x=4, dim_z=2)
                s.kf.F = np.eye(4)  # dt injected each predict
                s.kf.H = np.array([[1,0,0,0],[0,1,0,0]], float)
                s.kf.P = np.diag([50.0, 50.0, 2000.0, 2000.0])
                s.sigma_a = 160.0
                s.kf.R = np.diag([ (0.15*s.diam_sm)**2, (0.15*s.diam_sm)**2 ])
                s.kf.x[:2,0] = np.asarray(cen, float)
                s.kf.x[2:,0] = 0.0
                s.miss = 0
                s.prev_meas = np.array(cen, float)
                s.meas = np.array(cen, float)
                s.hit_streak = 1
                s.time_since_update = 0
                s.confirmed = False
                s.occluded = False
                s.occluded_age = 0
                s.merge_into = None
                s.parent_id = None
                s.merged_from_ids = None
                s.split_root_id   = None
                s.label_suffix    = ''
                s.child_ids = set()
                s.merge_streak = 0
                s.valid_visual = (p >= CLS_STAY)

            def _update_Q(s, dt):
                dt2 = dt*dt; dt3 = dt2*dt; dt4 = dt2*dt2
                q = s.sigma_a**2
                s.kf.Q = q * np.array([
                    [dt4/4,   0.0,   dt3/2,  0.0],
                    [0.0,   dt4/4,   0.0,   dt3/2],
                    [dt3/2,  0.0,    dt2,    0.0],
                    [0.0,   dt3/2,   0.0,    dt2]
                ])

            def pred(s, dt):
                s.kf.F[0,2] = dt
                s.kf.F[1,3] = dt
                s._update_Q(dt)
                s.kf.predict()
                s.miss += 1
                s.time_since_update += 1
                s.merged_from_ids = None

            def upd(s, cen, msk, dia, p, dt):
                s.prev_meas = s.meas.copy()
                s.meas = np.array(cen, float)
                chosen = choose_mask(s.mask, msk)
                s.mask = chosen
                area_px = float(np.count_nonzero(chosen))
                s.diam = 2.0 * np.sqrt(area_px / pi) if area_px > 0 else dia
                s.diam_sm = (1.0-EMA_D_ALPHA)*s.diam_sm + EMA_D_ALPHA*max(1e-6, s.diam)
                s.cnn_p = p
                s.valid_visual = (p >= CLS_STAY)
                s.miss = 0
                s.time_since_update = 0
                s.hit_streak += 1
                if not s.confirmed and s.hit_streak >= CONFIRM_HITS:
                    s.confirmed = True
                s.kf.R = np.diag([(0.16*s.diam_sm)**2, (0.16*s.diam_sm)**2])
                z = np.asarray(cen, dtype=float).reshape(2)
                s.kf.update(z)

            def state(s):  # [x,y,vx,vy]
                return s.kf.x.flatten()

        def make_label_text(tr):
            base = get_display_id(tr.split_root_id if tr.split_root_id is not None else tr.id)
            if tr.merged_from_ids and len(tr.merged_from_ids) >= 2 and tr.merge_streak >= MERGE_LABEL_MIN_STREAK:
                parts = []
                for rid in tr.merged_from_ids:
                    parts.append(str(get_display_id(rid)))
                parts = sorted(set(parts), key=lambda s: int(''.join(ch for ch in s if ch.isdigit())))[:2]
                if len(parts) >= 2:
                    return "ID " + "+".join(parts)
            if tr.label_suffix:
                return f"ID {base}{tr.label_suffix}"
            return f"ID {base}"

        # ---- Phase 3: per-frame detection, filtering, association, and track management ----
        while True:
            ok, frame = read_frame(fi)
            if not ok or frame is None:
                break

            raw_frame = frame.copy()
            # --- timing: physics for STILL, timestamps for SPIN ---
            if C['MODE'] == 'still':
                dt = 1.0 / fps_phys
                time_now = fi / fps_phys
            else:
                # spin mode will overwrite dt/time_now using timestamp later
                dt = 1.0 / max(1e-6, float(fps0))
                time_now = fi / max(1e-6, float(fps0))


            def ahead_xy(tr):
                x, y, vx, vy = tr.state()
                return x, y

            # Spin mode timing
            if C['MODE'] == 'spin':
                bidx, ib = divmod(fi, C['BURST'])
                if C['RESET'] and ib == 0:
                    tracks.clear()
                ts = read_timestamp_tpl(raw_frame)
                ts_cand = float(ts) if ts is not None else None
                
                if ts_cand is not None:
                    # There is timestamp from the frame therefore use it directly
                    if prev_ts is None:
                        prev_ts = ts_cand
                        dt_ms = 0.0             # first frame
                    else:
                        dt_ms = ts_cand - prev_ts
                        if dt_ms < 0:
                            dt_ms = 0.0
                        prev_ts = ts_cand
                else:
                    # OCR completely failed use minimal fallback to keep time moving
                    if dts:
                        dt_ms = dts[-1]
                        prev_ts += dt_ms
                    else:
                        dt_ms = 1000.0 / fps0
                        prev_ts = dt_ms
                
                dts.append(dt_ms)
                times.append(prev_ts)
                dt = max(1e-6, dt_ms / 1000.0)
                time_now = prev_ts / 1000.0


                if bidx % 2 == 1:
                    frame = cv2.flip(frame, 1)

            bar_x = find_center_bar_midpoint(frame)
            if bar_x is not None:
                if bar_ref_x is None:
                    bar_ref_x = bar_x

                dx = bar_ref_x - bar_x
                M = np.float32([[1, 0, dx],
                                    [0, 1, 0]])
                frame = cv2.warpAffine(frame, M, (W, H))

            # ------------- Detect --------------
            inst   = detector(frame)['instances'].to('cpu')
            boxes  = inst.pred_boxes.tensor.numpy()
            masks  = inst.pred_masks.numpy()
            scores = inst.scores.numpy()
            if len(scores) > 0:
                dprint(
                    f"RAW Mask-RCNN dets: {len(scores)}  | "
                    f"min score={scores.min():.3f} max={scores.max():.3f}"
                )
            else:
                dprint("RAW Mask-RCNN dets: 0  | no detections this frame")


            dc, dd, dm, dp, ds = [], [], [], [], []
            for bb, m, score in zip(boxes, masks, scores):
                if score < DET_SCORE_THRESH:
                    continue
                m_bool = m.astype(bool)
                M = cv2.moments(m.astype(np.uint8))
                if M['m00'] == 0:
                    continue
                cx = M['m10']/M['m00']; cy = M['m01']/M['m00']
                if cy < C['IGN_H']:
                    continue
                diam = area_to_diam(m_bool)
                if not (C['MIN_DIA_PX'] < diam < C['MAX_DIA_PX']):
                    continue
                roi = crop_center_square(frame, cx, cy)
                x = to_tensor_norm(roi).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits = classifier(x)
                    probs  = torch.softmax(logits, 1)
                    p_bub  = probs[0,0].item()     # adjust index if your model uses 1
                dc.append([cx, cy]); dd.append(diam); dm.append(m_bool); dp.append(p_bub); ds.append(float(score))
            dprint(f"After score/diam/IGN/CNN filter: {len(dc)}")

            # ---- DEBUG: per-detection info BEFORE NMS ----
            dprint(f"---- FRAME {fi} DET DEBUG ----")
            dprint("  RAW Mask-RCNN det count :", len(scores))
            dprint("  Filtered det count      :", len(dc))
            for idx, (cx, cy) in enumerate(dc):
                dprint(f"   DET {idx}: cx={cx:.1f}, cy={cy:.1f}, "
                      f"diam={dd[idx]:.1f}, p_bub={dp[idx]:.2f}, score={ds[idx]:.2f}")
            # ----------------------------------------------
                        
            # Drop near-duplicate detections
            if len(dm) > 1:
                rank = 0.5*np.asarray(ds) + 0.5*np.asarray(dp)
                keep = mask_pairwise_nms(dm, rank, iou_dup=0.82)
                if not np.all(keep):
                    dc = [c for c, k in zip(dc, keep) if k]
                    dd = [d for d, k in zip(dd, keep) if k]
                    dm = [m for m, k in zip(dm, keep) if k]
                    dp = [p for p, k in zip(dp, keep) if k]
                    ds = [s for s, k in zip(ds, keep) if k]

            # ---- DEBUG: after NMS ----
            dprint("  After mask-pairwise NMS:", len(dc))
            for idx, (cx, cy) in enumerate(dc):
                dprint(f"   NMS DET {idx}: cx={cx:.1f}, cy={cy:.1f}")
            # ---------------------------

            dprint(f"Passing into tracker assignment: {len(dc)} detections")
            
            # Split strong / weak 
            strong_idx = [i for i, s in enumerate(ds) if (s >= max(DET_SCORE_THRESH,0.55)) and (dp[i] >= CLS_ENTER)]
            weak_idx   = [i for i, s in enumerate(ds) if (s >= DET_SCORE_THRESH) and not (i in strong_idx)]
            dc_s = [dc[i] for i in strong_idx]
            dd_s = [dd[i] for i in strong_idx]
            dm_s = [dm[i] for i in strong_idx]
            dp_s = [dp[i] for i in strong_idx]

            # ---- DEBUG: split & track state BEFORE predict ----
            dprint("  Strong det idx:", strong_idx)
            dprint("  Weak det idx  :", weak_idx)
            dprint("  Tracks alive before predict:", len(tracks))
            for i, tr in enumerate(tracks):
                x, y, vx, vy = tr.state()
                dprint(f"   TRK {i}: tid={tr.id}, pred_before=({x:.1f},{y:.1f}), "
                      f"diam={tr.diam_sm:.1f}, miss={tr.miss}, confirmed={tr.confirmed}")
            # ---------------------------------------------------
                        
            dt_pred = dt
            for tr in tracks:
                tr.pred(dt_pred)

            debug_spawned = 0
            debug_reassoc = 0
            debug_killed  = 0
            debug_pairs = []

            # Association + handling
            if tracks and (dc_s or weak_idx):
                matched_trk = {}
                matched_det = {}
                used_trks, used_dets = set(), set()

                # Greedy 1↔1
                if len(tracks) == 1 and len(dc_s) == 1:
                    px, py, *_ = tracks[0].state()
                    cx, cy = dc_s[0]
                    if np.hypot(px - cx, py - cy) <= 2.0 * C['MAX_D']:
                        matched_trk[0] = strong_idx[0]
                        matched_det[strong_idx[0]] = 0
                        used_trks.add(0); used_dets.add(strong_idx[0])

                # Hungarian (normalized)
                if not matched_trk and tracks and dc_s:
                    Tn, Dn = len(tracks), len(dc_s)
                    dist_mat = np.zeros((Tn, Dn), float)
                    iou_mat  = np.zeros((Tn, Dn), float)
                    dia_ok   = np.zeros((Tn, Dn), dtype=bool)
                    for i, tr in enumerate(tracks):
                        px, py, _, _ = tr.state()
                        for j, (cx, cy) in enumerate(dc_s):
                            dist_mat[i, j] = np.hypot(px - cx, py - cy)
                            iou_mat[i, j]  = mask_iou(tr.mask, dm_s[j])
                    
                            if tr.hit_streak < 5:
                                dia_ok[i, j] = True
                            else:
                                dia_ok[i, j] = diameter_jump_ok(tr.diam_sm, dd_s[j])

                    overlapped_by_any_det = [False] * len(tracks)
                    for i in range(len(tracks)):
                        for j in range(Dn):
                            if iou_mat[i, j] >= IOU_MERGE_THRESH:
                                overlapped_by_any_det[i] = True
                                break
                    for i, tr in enumerate(tracks):
                        tr.merge_streak = (tr.merge_streak + 1) if overlapped_by_any_det[i] else 0

                    diam_list = [getattr(tr, 'diam_sm', 16.0) for tr in tracks]
                    char = max(12.0, 0.8 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                    cost = np.full((Tn, Dn), 1e6, float)
                    for i, tr in enumerate(tracks):
                        for j in range(Dn):
                            if not dia_ok[i, j]:
                                continue
                            m2 = mahalanobis2(tr, dc_s[j])
                            if (m2 <= GATE_MAHAL2):
                                cost[i, j] = (dist_mat[i, j] / char) + IOU_WEIGHT_DEFAULT*(1.0 - iou_mat[i, j])
                    ridx, cidx = linear_sum_assignment(cost)
                    for i, j in zip(ridx, cidx):
                        if cost[i, j] < 1e6 and dist_mat[i, j] <= C['MAX_D']:
                            matched_trk[i] = strong_idx[j]
                            matched_det[strong_idx[j]] = i
                            used_trks.add(i); used_dets.add(strong_idx[j])

                # Unmatched sets (all dets)
                unmatched_trk_idx = [i for i in range(len(tracks)) if i not in used_trks]
                unmatched_det_idx = [j for j in range(len(dc)) if j not in used_dets]

                # ---- DEBUG: association summary BEFORE merge/split ----
                dprint("  Matched track -> det mapping:")
                for ti, dj in matched_trk.items():
                    dprint(f"   Track idx {ti} (tid={tracks[ti].id}) <- DET {dj}")
                dprint("  Unmatched track idx:", unmatched_trk_idx)
                dprint("  Unmatched det idx  :", unmatched_det_idx)
                # -------------------------------------------------------

                # MERGE & SPLIT bookkeeping
                if dc:
                    Tn = len(tracks); Dn_all = len(dc)
                    iou_mat_all = np.zeros((Tn, Dn_all), float)
                    for i, tr in enumerate(tracks):
                        for j in range(Dn_all):
                            iou_mat_all[i, j] = mask_iou(tr.mask, dm[j])
                    # MERGE
                    for j in range(Dn_all):
                        overlappers = [i for i in range(Tn) if iou_mat_all[i, j] >= IOU_MERGE_THRESH]
                        if len(overlappers) >= 2:
                            if j in matched_det:
                                primary_i = matched_det[j]
                            else:
                                primary_i = max(overlappers, key=lambda i: iou_mat_all[i, j])
                                matched_trk[primary_i] = j
                                matched_det[j] = primary_i
                                used_trks.add(primary_i); used_dets.add(j)
                                if primary_i in unmatched_trk_idx: unmatched_trk_idx.remove(primary_i)
                                if j in unmatched_det_idx: unmatched_det_idx.remove(j)
                            primary = tracks[primary_i]
                            merged_set = set()
                            for i in overlappers:
                                merged_set.add(tracks[i].id)
                                if i != primary_i:
                                    tr = tracks[i]
                                    tr.occluded = True; tr.occluded_age = 0
                                    tr.merge_into = primary.id
                                    tr.parent_id = primary.id
                                    primary.child_ids.add(tr.id)
                            primary.merged_from_ids = merged_set
                    # SPLIT
                    id_to_index = {tr.id: idx for idx, tr in enumerate(tracks)}
                    for i in range(Tn):
                        overlapped_dets = [j for j in range(Dn_all) if iou_mat_all[i, j] >= IOU_SPLIT_THRESH]
                        if len(overlapped_dets) >= 2:
                            member_idxs = [i]
                            for cid in getattr(tracks[i], 'child_ids', set()):
                                idx = id_to_index.get(cid, None)
                                if idx is not None:
                                    member_idxs.append(idx)
                            seen = set(); member_idxs = [m for m in member_idxs if (m not in seen and not seen.add(m))]
                            diam_list = [getattr(tracks[k], 'diam_sm', 16.0) for k in member_idxs]
                            char_local = max(8.0, 0.5 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                            pairs = assign_to_members(member_idxs, overlapped_dets, dc, dm, tracks,
                                                      iou_weight=IOU_WEIGHT_DEFAULT, char_dist=char_local)
                            for ti, dj in pairs:
                                prev_d = matched_trk.get(ti, None)
                                if prev_d is not None:
                                    px, py, *_ = tracks[ti].state()
                                    pc = np.array([px, py]); c_prev = np.array(dc[prev_d]); c_new = np.array(dc[dj])
                                    prev_cost = np.hypot(*(pc - c_prev)) + IOU_WEIGHT_DEFAULT*(1.0 - mask_iou(tracks[ti].mask, dm[prev_d]))
                                    new_cost  = np.hypot(*(pc - c_new))  + IOU_WEIGHT_DEFAULT*(1.0 - mask_iou(tracks[ti].mask, dm[dj]))
                                    if new_cost >= prev_cost:
                                        continue
                                    used_dets.discard(prev_d)
                                    matched_det.pop(prev_d, None)
                                    if prev_d not in unmatched_det_idx:
                                        unmatched_det_idx.append(prev_d)
                                matched_trk[ti] = dj
                                matched_det[dj] = ti
                                used_trks.add(ti); used_dets.add(dj)
                                if dj in unmatched_det_idx: unmatched_det_idx.remove(dj)
                            if len(pairs) >= 2 and hasattr(tracks[i], 'child_ids'):
                                tracks[i].child_ids.clear()

                # collect debug pair metrics
                if C['SAVE_DEBUG'] and tracks and dc:
                    for ti, dj in matched_trk.items():
                        px, py, *_ = tracks[ti].state()
                        cx, cy = dc[dj]
                        d   = float(np.hypot(px - cx, py - cy))
                        iou = float(mask_iou(tracks[ti].mask, dm[dj]))
                        m2  = float(mahalanobis2(tracks[ti], dc[dj]))
                        debug_pairs.append({'frame': fi, 'track_id': tracks[ti].id, 'det_idx': dj,
                                            'd_px': d, 'iou': iou, 'mahal2': m2,
                                            'time_since_update': tracks[ti].time_since_update})

                # Update matched
                for i, j in matched_trk.items():
                    tr = tracks[i]
                    tr.occluded = False; tr.occluded_age = 0; tr.merge_into = None
                    tr.upd(dc[j], dm[j], dd[j], dp[j], dt_pred)
                dprint(f"Matched {len(matched_trk)} tracks this frame")

                # Re-associate to weak dets
                if weak_idx:
                    ut = [i for i in range(len(tracks)) if i not in used_trks and 0 < tracks[i].time_since_update <= MAX_COAST]
                    if ut:
                        dc_w = [dc[i] for i in weak_idx]; dm_w = [dm[i] for i in weak_idx]; dd_w = [dd[i] for i in weak_idx]
                        M, N = len(ut), len(dc_w)
                        cost_w = np.full((M, N), 1e6, float)
                        diam_list = [getattr(tracks[ti], 'diam_sm', 16.0) for ti in ut]
                        char = max(8.0, 0.5 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                        for a, ti in enumerate(ut):
                            tr = tracks[ti]
                            px, py, *_ = tr.state()
                            for b in range(N):
                                cx, cy = dc_w[b]
                                d = np.hypot(px - cx, py - cy)
                                iou = mask_iou(tr.mask, dm_w[b])
                                m2 = mahalanobis2(tr, dc_w[b])
                                if (m2 <= 2.0*GATE_MAHAL2) and (iou >= 0.02 or d <= 1.5*C['MAX_D']):
                                    cost_w[a, b] = (d / char) + IOU_WEIGHT_DEFAULT*(1.0 - iou)
                        rr, cc = linear_sum_assignment(cost_w)
                        for a, b in zip(rr, cc):
                            if cost_w[a, b] < 1e6 and dp[weak_idx[b]] >= 0.30:
                                ti = ut[a]; dj_global = weak_idx[b]
                                tr = tracks[ti]
                                tr.occluded = False; tr.occluded_age = 0; tr.merge_into = None
                                tr.upd(dc[dj_global], dm[dj_global], dd[dj_global], dp[dj_global], dt_pred)
                                used_trks.add(ti)
                                if dj_global in unmatched_det_idx:
                                    unmatched_det_idx.remove(dj_global)
                                debug_reassoc += 1

                # Re-associate unmatched dets
                if unmatched_det_idx:
                    cand_idxs = [i for i, tr in enumerate(tracks)
                                 if tr.time_since_update > 0 and tr.time_since_update <= MAX_COAST]
                    diam_list = [getattr(tracks[i], 'diam_sm', 16.0) for i in cand_idxs] if cand_idxs else []
                    char = max(8.0, 0.5 * float(np.nanmean(diam_list))) if len(diam_list) else 12.0
                    for j in list(unmatched_det_idx):
                        if dp[j] < 0.25:
                            continue
                        best_i, best_score = None, 1e9
                        for i in cand_idxs:
                            tr = tracks[i]
                            px, py, *_ = tr.state()
                            cx, cy = dc[j]
                            d = np.hypot(px - cx, py - cy)
                            iou = mask_iou(tr.mask, dm[j])
                            m2  = mahalanobis2(tr, dc[j])
                            if (m2 <= 2.0*GATE_MAHAL2) and (iou >= 0.02 or d <= 1.5*C['MAX_D']):
                                sc = (d / char) + IOU_WEIGHT_DEFAULT * (1.0 - iou)
                                if sc < best_score:
                                    best_score, best_i = sc, i
                        if best_i is not None:
                            tr = tracks[best_i]
                            tr.occluded = False; tr.occluded_age = 0; tr.merge_into = None
                            tr.upd(dc[j], dm[j], dd[j], dp[j], dt_pred)
                            used_trks.add(best_i)
                            unmatched_det_idx.remove(j)
                            debug_reassoc += 1

                # Controlled spawning
                MAX_SPAWN_PER_FRAME = 12
                SPAWN_SUPPRESS_FACTOR = 0.5  # how far from an existing bubble instance must be to allow a new track
                spawns = 0
                for j in unmatched_det_idx:
                    if dp[j] < CLS_ENTER:
                        continue
                    dj_cx, dj_cy = dc[j]
                    too_close = False
                
                    # if there is ANY nearby track, don't spawn a new one
                    for tr in tracks:
                        if tr.time_since_update <= MAX_COAST:  # don't require confirmed; just "alive"
                            cx, cy, *_ = tr.state()
                            d = np.hypot(cx - dj_cx, cy - dj_cy)
                            if d <= SPAWN_SUPPRESS_FACTOR * max(1.0, tr.diam_sm):
                                too_close = True
                                break
                
                    if too_close:
                        continue  # treat as a weird duplicate detection, not a new bubble
                
                    if spawns < MAX_SPAWN_PER_FRAME:
                        tracks.append(Trk(nxt, dc[j], dm[j], dd[j], dp[j]))
                        nxt += 1
                        spawns += 1
                        debug_spawned += 1

                # Age/prune
                alive = []
                for tr in tracks:
                    to_kill = False
                    if not tr.confirmed and (tr.miss > max(2, C['MAX_MIS'])):
                        to_kill = True
                    if tr.confirmed and tr.time_since_update > MAX_COAST:
                        to_kill = True
                    if tr.occluded and tr.occluded_age > MAX_OCCLUDE_AGE:
                        to_kill = True
                    if to_kill:
                        debug_killed += 1
                        continue
                    alive.append(tr)
                tracks = alive

                # Track-level NMS
                def tracks_pairwise_nms(tracks_list, iou_thr=0.86):
                    if len(tracks_list) <= 1:
                        return tracks_list
                    order = sorted(range(len(tracks_list)),
                                   key=lambda i: track_quality(tracks_list[i]),
                                   reverse=True)
                    keep_mask = [True]*len(tracks_list)
                    for a in range(len(order)):
                        i = order[a]
                        if not keep_mask[i]:
                            continue
                        for b in range(a+1, len(order)):
                            j = order[b]
                            if not keep_mask[j]:
                                continue
                            iou = mask_iou(tracks_list[i].mask, tracks_list[j].mask)
                            if iou >= iou_thr:
                                keep_mask[j] = False
                    return [t for t,k in zip(tracks_list, keep_mask) if k]
                tracks = tracks_pairwise_nms(tracks, iou_thr=0.86)

            else:
                # No tracks or no strong dets so spawn from all dets that pass enter
                for j in range(len(dc)):
                    if dp[j] >= CLS_ENTER:
                        tracks.append(Trk(nxt, dc[j], dm[j], dd[j], dp[j])); nxt += 1
                tracks = [tr for tr in tracks if tr.miss <= C['MAX_MIS']]
                debug_spawned = sum(1 for j in range(len(dc)) if dp[j] >= CLS_ENTER)

            # ---- Phase 4: write annotated outputs (filtered + optional debug videos) ----
            alpha = 0.4

            # v1: raw tracker IDs (debug)
            if vw1 is not None:
                v1 = frame.copy()
                for tr in tracks:
                    v1[tr.mask] = (v1[tr.mask]*(1 - alpha) + np.array(overlay_color)*alpha).astype(np.uint8)
                for tr in tracks:
                    ax, ay = ahead_xy(tr)
                    cx, cy = int(round(ax)), int(round(ay))
                    r = int(tr.diam_sm / 2)
                    cv2.circle(v1, (cx, cy), r, overlay_color, 2)
                    cv2.putText(v1, f"ID{tr.id}", (cx + 8, cy - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, overlay_color, 2)
                vw1.write(v1)

            # v2: raw ID + CNN prob (debug)
            if vw2 is not None:
                v2 = frame.copy()
                for tr in tracks:
                    ax, ay = ahead_xy(tr)
                    cx, cy = int(round(ax)), int(round(ay))
                    r = int(tr.diam_sm / 2)
                    col = (0, 255, 0) if tr.cnn_p >= CLS_STAY else (0, 0, 255)
                    cv2.rectangle(v2, (cx - r, cy - r), (cx + r, cy + r), col, 2)
                    cv2.putText(v2, f"ID{tr.id} p={tr.cnn_p:.2f}", (cx + 8, cy - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
                vw2.write(v2)

            # v3: filtered (representatives only) — display IDs
            filtered = frame.copy()
            
            # --- only draw tracks that were actually matched this frame ---
            draw_tracks = [tr for tr in tracks if tr.confirmed 
                                                  and tr.valid_visual 
                                                  and tr.miss == 0         # MUST have a detection this frame
                                                  and not tr.occluded]
            
            # ---- DEBUG: which tracks are actually drawn ----
            dprint("  Draw-track tids this frame:", [tr.id for tr in draw_tracks])
            dprint(f"Drawing {len(draw_tracks)} bubbles this frame")
            
            # --- draw overlays ---
            for tr in draw_tracks:
                filtered[tr.mask] = (filtered[tr.mask]*(1 - alpha) + np.array(overlay_color)*alpha).astype(np.uint8)
            
            for tr in draw_tracks:
                ax, ay = ahead_xy(tr)
                cx, cy = int(round(ax)), int(round(ay))
                r = int(tr.diam_sm / 2)
                cv2.circle(filtered, (cx, cy), r, overlay_color, 2)
                disp = get_display_id(tr.id)
                cv2.putText(filtered, f"ID {disp}", (cx + 8, cy - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, overlay_color, 2)
            
            vw3.write(filtered)


            # v4: DEBUG OVERLAY
            if vw4 is not None:
                v4 = frame.copy()
                # A) detections
                for j, (cx, cy) in enumerate(dc):
                    cv2.circle(v4, (int(cx), int(cy)), 6, (255, 100, 0), 2)
                    cv2.putText(v4, f"D{j}", (int(cx)+7, int(cy)-7), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,100,0), 1)
                # B) tracks prediction + id
                for i, tr in enumerate(tracks):
                    x, y, vx, vy = tr.state()
                    cv2.drawMarker(v4, (int(x), int(y)), (255,255,255),
                                   markerType=cv2.MARKER_TILTED_CROSS, markerSize=14, thickness=2)
                    M = cv2.moments(tr.mask.astype(np.uint8))
                    if M['m00'] > 0:
                        mx, my = int(M['m10']/M['m00']), int(M['m01']/M['m00'])
                        x, y, vx, vy = tr.state()
                        dprint(f"ID {tr.id}: track center=({x:.1f},{y:.1f}), "
                              f"mask centroid=({mx},{my}), diam={tr.diam_sm:.1f}")

                        color = (0,255,0) if tr.confirmed else (0,0,255)
                        cv2.circle(v4, (mx, my), max(2, int(tr.diam_sm/2)), color, 1)
                    tag = f"T{tr.id} hit={tr.hit_streak} miss={tr.miss} tsu={tr.time_since_update}"
                    cv2.putText(v4, tag, (int(x)+8, int(y)+16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)
                # C) matched metrics text (optional)
                hud = [
                    f"frame={fi} dets={len(dc)} tracks={len(tracks)}",
                    f"spawned={debug_spawned} re-assoc={debug_reassoc} killed={debug_killed}",
                    f"MAX_D={C['MAX_D']} GATE_M2={GATE_MAHAL2} IOU_W={IOU_WEIGHT_DEFAULT}"
                ]
                y0 = 24
                for line in hud:
                    cv2.putText(v4, line, (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)
                    y0 += 22
                vw4.write(v4)

            # --- write binary mask frame ---
            mask_u8 = np.zeros((H, W), dtype=np.uint8)
            for tr in draw_tracks:
                mask_u8[tr.mask] = 255  # union mask
            
            # mp4v writer expects 3-channel BGR
            mask_bgr = cv2.cvtColor(mask_u8, cv2.COLOR_GRAY2BGR)
            vw_mask.write(mask_bgr)


            # CSV rows: all confirmed & visually valid tracks (use display id)
            for tr in tracks:
                if tr.confirmed and tr.valid_visual and tr.miss == 0 and not tr.occluded:
                    kx, ky, kvx, kvy = tr.state()
                    mx, my = tr.meas  # raw measurement centroid from detection
            
                    disp_id = get_display_id(tr.id)
                    rows.append({
                        'frame': fi, 'time_s': time_now,
                        'id': disp_id,
            
                        'cx': float(mx),
                        'cy': float(my),
            
                        # keep diam as you already do
                        'diam_px': tr.diam_sm,
                        'cnn_p': tr.cnn_p,
                    })


            if C['SAVE_DEBUG'] and debug_pairs:
                debug_rows.extend(debug_pairs)

            fi += 1; self.prog['value'] = fi; self.update_idletasks()

        # tidy up writers
        if cap is not None:
            cap.release()
        vw3.release()
        if vw1: vw1.release()
        if vw2: vw2.release()
        if vw4: vw4.release()
        if vw_mask: vw_mask.release()


        # ---- Phase 5: export measurement CSVs and compute dimensionless parameters (if scale provided) ----
        if C['SAVE_DEBUG'] and debug_rows:
            pd.DataFrame(debug_rows).to_csv(
                os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_debug_pairs.csv"),
                index=False
            )

        if C['SAVE_CSVS'] and rows:
            df = pd.DataFrame(rows).sort_values(['id', 'time_s'])

            G = df.groupby('id', sort=False)
            
            if C.get("INPUT_TYPE") == "tif":
                # 2-frame displacement to suppress 0, jump, 0, jump quantization
                dt2 = G['time_s'].diff(2)
                df['vel_x_px_s'] = G['cx'].diff(2) / dt2
                df['vel_y_px_s'] = G['cy'].diff(2) / dt2
            else:
                dt1 = G['time_s'].diff(1)
                df['vel_x_px_s'] = G['cx'].diff(1) / dt1
                df['vel_y_px_s'] = G['cy'].diff(1) / dt1
            
            df['vel_px_s'] = np.hypot(df['vel_x_px_s'], df['vel_y_px_s'])
            
            # clean up non-finite
            for c in ['vel_x_px_s','vel_y_px_s','vel_px_s']:
                df[c] = pd.to_numeric(df[c], errors='coerce')
                df.loc[~np.isfinite(df[c]), c] = np.nan


            # Unit scaling + dimensionless numbers (if scale provided)
            scale = C.get('SCALE_IN_PER_PX')
            if scale:
                df['diam_in']    = df['diam_px'] * scale
                df['vel_x_in_s'] = df['vel_x_px_s'] * scale
                df['vel_y_in_s'] = df['vel_y_px_s'] * scale
                df['vel_in_s']   = df['vel_px_s']   * scale

                gc = C['G_C']
                U = df['vel_in_s']
                L = df['diam_in']
                if 'diam_in' in df.columns:
                    df['Re'] = (rho_l * U * L) / (mu_l * gc)
                    df['Eo'] = (delta_r * g_eff * (L**2)) / (sigma * gc)
                    df['We'] = (rho_l * (U**2) * L) / (sigma * gc)
                    df['Fr'] = (U**2) / (g_eff * L)
                    df['Ga'] = (rho_l * L * np.sqrt(g_eff * L)) / (mu_l * gc)
                    df['Mo'] = (g_eff * (mu_l * gc)**4 * (delta_r)) / ((rho_l**2) * (sigma * gc)**3)
                    df['Bo'] = df['Eo']

            # Write main track CSV
            path_trk = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_tracked.csv")
            df.to_csv(path_trk, index=False, na_rep='', float_format='%.15g')

            # Helper to compute numeric average from a CSV column (avoids NaNs/strings)
            def excel_average_from_csv_column(csv_path, colname):
                values = []
                with open(csv_path, newline='', encoding='utf-8') as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        s = (row.get(colname, '') or '').strip()
                        if s == '':
                            continue
                        try:
                            v = float(s)
                        except Exception:
                            continue
                        if math.isfinite(v):
                            values.append(v)
                return float(sum(values)/len(values)) if values else float('nan')

            # Per-ID and overall averages
            df_num = pd.read_csv(path_trk)
            cols_all = [c for c in [
                'vel_x_px_s','vel_y_px_s','vel_px_s','diam_px',
                'vel_x_in_s','vel_y_in_s','vel_in_s','diam_in',
                'Re','Eo','We','Fr','Ga','Mo','Bo'
            ] if c in df_num.columns]

            for c in cols_all:
                df_num[c] = pd.to_numeric(df_num[c], errors='coerce')

            per_id = df_num.groupby('id', sort=True)[cols_all].mean().reset_index()
            per_id = per_id.rename(columns={c: f'avg_{c}' for c in cols_all})

            overall = {'id': 'Averages'}
            for c in cols_all:
                overall[f'avg_{c}'] = excel_average_from_csv_column(path_trk, c)

            summary = pd.concat([pd.DataFrame([overall]), per_id], ignore_index=True)
            summary = summary.reindex(columns=['id'] + [f'avg_{c}' for c in cols_all])

            mdot_col = 'avg_mdot_gas_lbm_s'
            summary[mdot_col] = np.nan 
            
            try:
                # Must have physical diameter for real mass flow
                if 'diam_in' in df_num.columns and df_num['diam_in'].notna().any():
                    # Plane location in pixels
                    plane_y = float(COUNT_PLANE_Y_FRAC) * float(H)

                    # Video duration (seconds) from the actual timestamps
                    t0 = float(df_num['time_s'].min())
                    t1 = float(df_num['time_s'].max())
                    T = max(1e-12, (t1 - t0))

                    # Ensure sorted within each ID
                    df_sorted = df_num.sort_values(['id', 'time_s']).copy()

                    # Previous cy per ID
                    df_sorted['cy_prev'] = df_sorted.groupby('id')['cy'].shift(1)

                    # Define crossing condition
                    if COUNT_DIRECTION.lower() == "up":
                        # rising: cy decreases across plane
                        cross = (df_sorted['cy_prev'] > plane_y) & (df_sorted['cy'] <= plane_y)
                    elif COUNT_DIRECTION.lower() == "down":
                        # falling: cy increases across plane
                        cross = (df_sorted['cy_prev'] < plane_y) & (df_sorted['cy'] >= plane_y)
                    else:
                        # both directions
                        cross_up = (df_sorted['cy_prev'] > plane_y) & (df_sorted['cy'] <= plane_y)
                        cross_dn = (df_sorted['cy_prev'] < plane_y) & (df_sorted['cy'] >= plane_y)
                        cross = cross_up | cross_dn

                    # Keep only rows where a crossing occurs
                    df_cross = df_sorted.loc[cross].copy()

                    if not df_cross.empty:
                        # Count each ID once: take FIRST crossing per ID
                        df_first = df_cross.groupby('id', sort=True).head(1)

                        # Diameter at crossing (fallback to median per ID if missing)
                        med_diam = df_sorted.groupby('id', sort=True)['diam_in'].median()
                        d = df_first['diam_in'].copy()
                        d = d.fillna(df_first['id'].map(med_diam))

                        # Drop any IDs still missing diameter
                        d = d[pd.notna(d) & np.isfinite(d) & (d > 0)]

                        if len(d) > 0:
                            # Sphere volume [in^3]
                            vol_in3 = (math.pi / 6.0) * (d ** 3)

                            # Mass per bubble [lbm]
                            mass_lbm = rho_g * vol_in3

                            # Overall mdot [lbm/s]
                            mdot_overall = float(mass_lbm.sum() / T)

                            summary.loc[0, mdot_col] = mdot_overall

            except Exception:
                # Leave blank instead of breaking exports
                pass

            # Force mdot column to be the LAST column
            cols_now = [c for c in summary.columns if c != mdot_col] + [mdot_col]
            summary = summary.reindex(columns=cols_now)
            
            path_avg = os.path.join(C['OUT_DIR'], f"{C['PREFIX']}_averages.csv")

            settings_rows = [
                ['# BubbleTracker GUI Settings'],
                ['VIDEO',                 C['VIDEO']],
                ['MASK_WEIGHTS',          C['MASK_W']],
                ['CNN_WEIGHTS',           C['CNN_W']],
                ['MODE',                  C['MODE']],
                ['FPS_STILL',             C['FPS_STILL']],
                ['CAP_FPS_VIDEO',         cap_fps],                  
                ['MAX_ASSIGN_DIST_PX',    C['MAX_D']],
                ['MAX_MISSES_FRAMES',     C['MAX_MIS']],
                ['IGNORE_TEXT_PX',        C['IGN_H']],
                ['MIN_DIAM_PX',           C['MIN_DIA_PX']],
                ['MAX_DIAM_PX',           C['MAX_DIA_PX']],
                ['SCALE_IN_INPUT',        C.get('SCALE_IN_RAW')],    # inches input from GUI
                ['SCALE_PX_INPUT',        C.get('SCALE_PX_RAW')],    # pixel input from GUI
                ['SCALE_IN_PER_PX',       C.get('SCALE_IN_PER_PX')],
                ['LIQUID',                C['LIQ']],
                ['GAS',                   C['GAS']],
                ['RPM',                   C['RPM']],
                ['RADIUS_IN',             C['RADIUS_IN']],
                ['FRAMES_PER_BURST',      C['BURST']],
                ['RESET_IDS_PER_BURST',   C['RESET']],
                ['SAVE_DEBUG',            C['SAVE_DEBUG']],
                ['SAVE_MEASUREMENT_CSVS', C['SAVE_CSVS']],
            ]
            
            with open(path_avg, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                for row in settings_rows:
                    writer.writerow(row)
                writer.writerow([])  # blank line
            
                summary.to_csv(f, index=False, float_format='%.15g')



# --- end class BubbleTrackerApp ---
if __name__ == '__main__':
    BubbleTrackerApp().mainloop()


ModuleNotFoundError: No module named 'cv2'